In [5]:
import sys
sys.path.append('..')

In [3]:
from src.data.load import load_dataset
from src.data.split import temporal_split, random_split
import glob, os, pandas as pd
from src.config import ID_FOLDER

dataset_v2 = load_dataset()

csv_paths = glob.glob(os.path.join(ID_FOLDER, "*", "*_clean.csv"))
dfs = []
for p in csv_paths:
    year = os.path.basename(os.path.dirname(p))
    df_year = pd.read_csv(p, low_memory=False)
    df_year["year_folder"] = year
    dfs.append(df_year)
isolates_v2 = pd.concat(dfs, ignore_index=True)

split_temporal = temporal_split(dataset_v2, isolates_v2)
split_random = random_split(dataset_v2)

print("n_samples:", dataset_v2.n_samples)
print("Train:", split_temporal['train_idx'].shape, "Test:", split_temporal['test_idx'].shape)

temporal_split: train=3331, test=1313
random_split: train=3483, test=1161
n_samples: 4644
Train: (3331,) Test: (1313,)


In [4]:
spectrum = dataset_v2.X[0]

print("shape:", spectrum.shape)
print("dtype:", spectrum.dtype)
print("n_peaks:", spectrum.n_peaks)

print("\nintensities shape:", spectrum.intensities.shape)
print("intensities sample:", spectrum.intensities[:10])

print("\nmass_to_charge_ratios shape:", spectrum.mass_to_charge_ratios.shape)
print("mass_to_charge_ratios sample:", spectrum.mass_to_charge_ratios[:10])

shape: (6000, 2)
dtype: float64
n_peaks: 6000

intensities shape: (6000,)
intensities sample: [0.00034792 0.00048636 0.00013401 0.00072679 0.0012926  0.00101188
 0.00021698 0.00017965 0.00014184 0.00030657]

mass_to_charge_ratios shape: (6000,)
mass_to_charge_ratios sample: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]


In [5]:
import importlib
from src.data import features
importlib.reload(features)
from src.data.features import to_feature_matrix

X = to_feature_matrix(dataset_v2)
print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("Sample row (first 10 values):", X[0, :10])

X shape: (4644, 6000)
X dtype: float64
Sample row (first 10 values): [0.00034792 0.00048636 0.00013401 0.00072679 0.0012926  0.00101188
 0.00021698 0.00017965 0.00014184 0.00030657]


In [6]:
import pandas as pd
from src.data.load import load_dataset, load_metadata_with_years

dataset_v3 = load_dataset()
metadata_years = load_metadata_with_years()

# Build a dataframe: code, year_folder, and each antibiotic's label
merged = dataset_v3.y.merge(metadata_years, on='code', how='left')

print("R rate per year, per antibiotic:\n")
for ab in ['Ciprofloxacin', 'Cotrimoxazole', 'Ceftriaxone', 
           'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin']:
    print(f"\n{ab}:")
    print(merged.groupby('year_folder')[ab].mean().round(3))

R rate per year, per antibiotic:


Ciprofloxacin:
year_folder
2015    0.315
2016    0.317
2017    0.284
2018    0.273
Name: Ciprofloxacin, dtype: object

Cotrimoxazole:
year_folder
2015    0.393
2016    0.336
2017    0.322
2018    0.316
Name: Cotrimoxazole, dtype: object

Ceftriaxone:
year_folder
2015    0.225
2016    0.241
2017    0.204
2018    0.183
Name: Ceftriaxone, dtype: object

Amoxicillin-Clavulanic acid:
year_folder
2015    0.337
2016    0.257
2017     0.21
2018    0.285
Name: Amoxicillin-Clavulanic acid, dtype: object

Ampicillin-Amoxicillin:
year_folder
2015    0.573
2016    0.589
2017    0.573
2018    0.589
Name: Ampicillin-Amoxicillin, dtype: object


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from src.data.features import to_feature_matrix

X = to_feature_matrix(dataset_v3)

# Binary label: 1 if 2018, 0 if 2015-2017
years = merged['year_folder'].astype(int)
y_year = (years == 2018).astype(int).to_numpy()

print("Class balance (fraction 2018):", y_year.mean())

X_train, X_test, y_train, y_test = train_test_split(
    X, y_year, test_size=0.25, random_state=42, stratify=y_year
)

model = XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, y_proba)
print(f"\nYear-classifier AUROC (2018 vs 2015-2017): {auroc:.4f}")
print("(AUROC near 0.5 = no detectable spectral shift; near 1.0 = strong shift)")

Class balance (fraction 2018): 0.2827304048234281

Year-classifier AUROC (2018 vs 2015-2017): 0.9571
(AUROC near 0.5 = no detectable spectral shift; near 1.0 = strong shift)


In [8]:
import numpy as np
from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.config import RANDOM_SEED

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

OPTION_A = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
            'Ciprofloxacin', 'Cotrimoxazole']
OPTION_D = ['Ceftriaxone', 'Ciprofloxacin', 'Ampicillin-Amoxicillin',
            'Cotrimoxazole', 'Amoxicillin-Clavulanic acid']

for order, name in [(OPTION_A, 'Option A'), (OPTION_D, 'Option D')]:
    Y = np.column_stack([dataset.to_numpy(ab) for ab in order])
    X_train, X_test = X[split['train_idx']], X[split['test_idx']]
    Y_train, Y_test = Y[split['train_idx']], Y[split['test_idx']]

    chain = ClassifierChain(
        estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
        order=list(range(len(order))),
        random_state=RANDOM_SEED,
    )
    chain.fit(X_train, Y_train)
    Y_pred = chain.predict(X_test)

    all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
    all_zero_true_frac = (Y_test.sum(axis=1) == 0).mean()
    print(f"{name}: fraction all-zero predictions = {all_zero_frac:.4f} "
          f"(true all-zero fraction = {all_zero_true_frac:.4f})")

temporal_split: train=3331, test=1313
Option A: fraction all-zero predictions = 0.3899 (true all-zero fraction = 0.3641)
Option D: fraction all-zero predictions = 0.6535 (true all-zero fraction = 0.3641)


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split
from src.evaluation.metrics import multilabel_metrics
from src.config import RANDOM_SEED

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

order = ['Ciprofloxacin', 'Ceftriaxone', 'Ampicillin-Amoxicillin',
         'Cotrimoxazole', 'Amoxicillin-Clavulanic acid']

Y = np.column_stack([dataset.to_numpy(ab) for ab in order])

# Flip Ciprofloxacin's labels (column 0) — R becomes majority, everything else unchanged
print("Original Ciprofloxacin R rate:", Y[:, 0].mean())
Y[:, 0] = 1 - Y[:, 0]
print("Flipped Ciprofloxacin R rate:", Y[:, 0].mean())

train_idx, test_idx = split['train_idx'], split['test_idx']
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

Y_pred = chain.predict(X_test)
Y_proba = chain.predict_proba(X_test)

for i, ab in enumerate(order):
    label = f"{ab} (FLIPPED)" if i == 0 else ab
    auroc = roc_auc_score(Y_test[:, i], Y_proba[:, i])
    f1 = f1_score(Y_test[:, i], Y_pred[:, i])
    print(f"{label}: AUROC={auroc:.4f}, F1={f1:.4f}")

joint_metrics = multilabel_metrics(Y_test, Y_pred)
all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
print(f"\njoint metrics: {joint_metrics}")
print(f"all_zero_frac: {all_zero_frac:.4f}")

temporal_split: train=3331, test=1313
Original Ciprofloxacin R rate: 0.2904823428079242
Flipped Ciprofloxacin R rate: 0.7095176571920758


KeyboardInterrupt: 

In [ ]:
order_f = ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin',
           'Cotrimoxazole', 'Ampicillin-Amoxicillin']

Y = np.column_stack([dataset.to_numpy(ab) for ab in order_f])

X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order_f))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

Y_pred = chain.predict(X_test)
Y_proba = chain.predict_proba(X_test)

for i, ab in enumerate(order_f):
    auroc = roc_auc_score(Y_test[:, i], Y_proba[:, i])
    f1 = f1_score(Y_test[:, i], Y_pred[:, i])
    print(f"{ab}: AUROC={auroc:.4f}, F1={f1:.4f}")

joint_metrics = multilabel_metrics(Y_test, Y_pred)
all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
print(f"\njoint metrics: {joint_metrics}")
print(f"all_zero_frac: {all_zero_frac:.4f}")

Ceftriaxone: AUROC=0.8617, F1=0.5539
Amoxicillin-Clavulanic acid: AUROC=0.5951, F1=0.1042
Ciprofloxacin: AUROC=0.7585, F1=0.4256
Cotrimoxazole: AUROC=0.6512, F1=0.2192
Ampicillin-Amoxicillin: AUROC=0.6065, F1=0.3480

joint metrics: {'hamming_loss': 0.27844630616907845, 'jaccard_score': 0.09058136582889058}
all_zero_frac: 0.8378


c:\Admin - Vaishali\Academics\VITV\Project_4_1\Implementation\amr_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
option_a_random = pd.read_csv('../results/metrics/chains_random_seeds.csv')
option_a_zero = option_a_random[option_a_random['order_name'] == 'option_a']['all_zero_frac']
print("Option A all_zero_frac across 10 seeds:", option_a_zero.tolist())
print("Mean:", option_a_zero.mean(), "Std:", option_a_zero.std())

Option A all_zero_frac across 10 seeds: [0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.2901751713632902, 0.2901751713632902, 0.2901751713632902,

In [ ]:
order = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
         'Ciprofloxacin', 'Cotrimoxazole']

Y_full = np.column_stack([dataset.to_numpy(ab) for ab in order])
Y_test_true = Y_full[test_idx]

true_all_zero_frac = (Y_test_true.sum(axis=1) == 0).mean()
print(f"True all-zero fraction (temporal test set): {true_all_zero_frac:.4f}")
print(f"n={len(Y_test_true)}, all-zero count={int((Y_test_true.sum(axis=1) == 0).sum())}")

True all-zero fraction (temporal test set): 0.3641
n=1313, all-zero count=478


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

# quick synthetic check: does predict_proba return columns in original Y order,
# regardless of the chain's internal `order`?
X_dummy = np.random.rand(50, 5)
Y_dummy = np.random.randint(0, 2, size=(50, 3))  # 3 labels, easy to distinguish

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[2, 0, 1], random_state=0)
chain.fit(X_dummy, Y_dummy)

print("chain.order_:", chain.order_)  # the actual fitting order used
pred = chain.predict(X_dummy[:5])
print("predict() output shape:", pred.shape)  # should be (5, 3) - matching Y_dummy's original column count/order

chain.order_: [2, 0, 1]
predict() output shape: (5, 3)


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

X_dummy = np.random.rand(200, 5)

# Make 3 labels that are EASY to tell apart and trivially learnable from X_dummy
Y_dummy = np.column_stack([
    (X_dummy[:, 0] > 0.5).astype(int),   # label 0: depends only on X col 0
    (X_dummy[:, 1] > 0.5).astype(int),   # label 1: depends only on X col 1
    (X_dummy[:, 2] > 0.5).astype(int),   # label 2: depends only on X col 2
])

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[2, 0, 1], random_state=0)
chain.fit(X_dummy, Y_dummy)

pred = chain.predict(X_dummy)

# If column order is correct, predicted column i should closely match Y_dummy column i
for i in range(3):
    accuracy = (pred[:, i] == Y_dummy[:, i]).mean()
    print(f"Column {i}: accuracy against Y_dummy column {i} = {accuracy:.3f}")

Column 0: accuracy against Y_dummy column 0 = 1.000
Column 1: accuracy against Y_dummy column 1 = 1.000
Column 2: accuracy against Y_dummy column 2 = 1.000


In [ ]:
# Quick check: is the temporal Option A vs baseline gap comparable to random-split noise?

baseline_temporal_jaccard = 0.2323
option_a_temporal_jaccard = 0.2453
temporal_gap = option_a_temporal_jaccard - baseline_temporal_jaccard

# reference noise magnitude from random split (10 seeds each)
baseline_random_std = 0.0047   # from earlier: 0.273 ± 0.005
option_a_random_std = 0.0063   # from earlier: 0.291 ± 0.006
pooled_std = (baseline_random_std**2 + option_a_random_std**2) ** 0.5

print(f"Temporal gap: {temporal_gap:.4f}")
print(f"Reference pooled std (from random-split seed variance): {pooled_std:.4f}")
print(f"Gap as multiple of reference std: {temporal_gap / pooled_std:.2f}")

Temporal gap: 0.0130
Reference pooled std (from random-split seed variance): 0.0079
Gap as multiple of reference std: 1.65


In [ ]:
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
import numpy as np

X_dummy = np.random.rand(20, 5)
Y_dummy = np.random.randint(0, 2, size=(20, 3))

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[0, 1, 2], random_state=0)
chain.fit(X_dummy, Y_dummy)

# Inspect the actual estimators inside the chain
for i, est in enumerate(chain.estimators_):
    print(f"Step {i}: n_features expected = {est.n_features_in_}")

Step 0: n_features expected = 5
Step 1: n_features expected = 6
Step 2: n_features expected = 7


In [ ]:
import sys
sys.path.append('..')

from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

print("n_samples:", dataset.n_samples)
print("X shape:", X.shape)

temporal_split: train=3331, test=1313
n_samples: 4644
X shape: (4644, 6000)


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.config import RANDOM_SEED

OPTION_A = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
            'Ciprofloxacin', 'Cotrimoxazole']

Y = np.column_stack([dataset.to_numpy(ab) for ab in OPTION_A])
X_train, X_test = X[split['train_idx']], X[split['test_idx']]
Y_train = Y[split['train_idx']]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(OPTION_A))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

# small subsample for a quick check
X_sub = X_test[:50]

# manual recursion, same logic as shap_chain
prior_pred_columns = []
for i in range(len(OPTION_A)):
    aug_sub = np.hstack([X_sub] + prior_pred_columns) if prior_pred_columns else X_sub
    pred_i = chain.estimators_[i].predict(aug_sub)
    prior_pred_columns.append(pred_i.reshape(-1, 1))

manual_preds = np.hstack(prior_pred_columns)  # shape (50, 5), in chain order (0..4)

# official chain.predict on same subsample - note this returns columns in 
# ORIGINAL Y order (order=[0,1,2,3,4] here so it's identical to chain order anyway)
official_preds = chain.predict(X_sub)

print("Manual recursion matches chain.predict() exactly:", np.array_equal(manual_preds, official_preds))

Manual recursion matches chain.predict() exactly: True


In [ ]:
import numpy as np

baseline_full = np.load('../results/metrics/shap_baseline_temporal_full.npz')
chain_full = np.load('../results/metrics/shap_chain_option_a_temporal_full.npz')

for ab in ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'Cotrimoxazole']:
    baseline_top15 = set(np.argsort(baseline_full[ab])[::-1][:15])
    chain_top15 = set(np.argsort(chain_full[ab])[::-1][:15])
    overlap = len(baseline_top15 & chain_top15)
    print(f"{ab}: {overlap}/15 top spectral bins overlap between baseline and chain")

Ceftriaxone: 10/15 top spectral bins overlap between baseline and chain
Amoxicillin-Clavulanic acid: 7/15 top spectral bins overlap between baseline and chain
Ciprofloxacin: 6/15 top spectral bins overlap between baseline and chain
Cotrimoxazole: 2/15 top spectral bins overlap between baseline and chain


In [ ]:
import numpy as np
from xgboost import XGBClassifier
import shap

X_train, X_test = X[split['train_idx']], X[split['test_idx']]
X_sub = X_test[:750]  # same subsample size as before, doesn't need to match exact indices for this noise check

antibiotics_to_check = ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'Cotrimoxazole']

for ab in antibiotics_to_check:
    y = dataset.to_numpy(ab)
    y_train = y[split['train_idx']]

    model_seed1 = XGBClassifier(random_state=42, eval_metric='logloss')
    model_seed1.fit(X_train, y_train)
    shap1 = shap.TreeExplainer(model_seed1).shap_values(X_sub)
    if isinstance(shap1, list):
        shap1 = shap1[1] if len(shap1) > 1 else shap1[0]
    top15_seed1 = set(np.argsort(np.abs(shap1).mean(axis=0))[::-1][:15])

    model_seed2 = XGBClassifier(random_state=123, eval_metric='logloss')
    model_seed2.fit(X_train, y_train)
    shap2 = shap.TreeExplainer(model_seed2).shap_values(X_sub)
    if isinstance(shap2, list):
        shap2 = shap2[1] if len(shap2) > 1 else shap2[0]
    top15_seed2 = set(np.argsort(np.abs(shap2).mean(axis=0))[::-1][:15])

    overlap = len(top15_seed1 & top15_seed2)
    print(f"{ab}: {overlap}/15 top-15 overlap between two seeds (baseline, same data)")

Ceftriaxone: 15/15 top-15 overlap between two seeds (baseline, same data)
Amoxicillin-Clavulanic acid: 15/15 top-15 overlap between two seeds (baseline, same data)
Ciprofloxacin: 15/15 top-15 overlap between two seeds (baseline, same data)
Cotrimoxazole: 15/15 top-15 overlap between two seeds (baseline, same data)


In [ ]:
order_g = ['Ampicillin-Amoxicillin', 'Cotrimoxazole', 'Ciprofloxacin',
           'Amoxicillin-Clavulanic acid', 'Ceftriaxone']

from src.interpretability.shap_analysis import shap_chain
chain_g_results = shap_chain(X, dataset, split, 'temporal', order_g, 'option_g', subsample_n=750)

[shap_chain:option_g] fitting chain...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): building augmented input...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): running TreeExplainer on (750, 6000)...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): fraction_prior_attribution=0.0000
[shap_chain:option_g] step 1 (Cotrimoxazole): building augmented input...
[shap_chain:option_g] step 1 (Cotrimoxazole): running TreeExplainer on (750, 6001)...
[shap_chain:option_g] step 1 (Cotrimoxazole): fraction_prior_attribution=0.1040
[shap_chain:option_g] step 2 (Ciprofloxacin): building augmented input...
[shap_chain:option_g] step 2 (Ciprofloxacin): running TreeExplainer on (750, 6002)...
[shap_chain:option_g] step 2 (Ciprofloxacin): fraction_prior_attribution=0.0923
[shap_chain:option_g] step 3 (Amoxicillin-Clavulanic acid): building augmented input...
[shap_chain:option_g] step 3 (Amoxicillin-Clavulanic acid): running TreeExplainer on (750, 6003)...
[shap_chain:option_g] step

In [ ]:
baseline_full = np.load('../results/metrics/shap_baseline_temporal_full.npz')

for ab in ['Cotrimoxazole', 'Ciprofloxacin', 'Amoxicillin-Clavulanic acid', 'Ceftriaxone']:
    baseline_top15 = set(np.argsort(baseline_full[ab])[::-1][:15])
    chain_g_top15 = set(chain_g_results[ab]['top15_spectral_feature_indices'])
    overlap = len(baseline_top15 & chain_g_top15)
    position = chain_g_results[ab]['chain_position']
    print(f"{ab} (position {position}): {overlap}/15 top spectral bins overlap with baseline")

Cotrimoxazole (position 1): 3/15 top spectral bins overlap with baseline
Ciprofloxacin (position 2): 8/15 top spectral bins overlap with baseline
Amoxicillin-Clavulanic acid (position 3): 6/15 top spectral bins overlap with baseline
Ceftriaxone (position 4): 9/15 top spectral bins overlap with baseline


In [11]:
from src.data.load import load_metadata_with_years
import glob, os
import pandas as pd
from src.config import ID_FOLDER

# load_metadata_with_years() only returns code + year_folder (trimmed),
# but we need the FULL metadata with all antibiotic columns here,
# so rebuild it directly the same way load_metadata_with_years does internally
paths = glob.glob(os.path.join(ID_FOLDER, '*', '*_clean.csv'))
frames = []
for path in paths:
    year_folder = os.path.basename(os.path.dirname(path))
    df = pd.read_csv(path, low_memory=False)
    df['year_folder'] = year_folder
    frames.append(df)
metadata = pd.concat(frames, ignore_index=True)

print("metadata shape:", metadata.shape)

metadata shape: (111257, 93)


In [12]:
import pandas as pd

target_species = ['Staphylococcus aureus', 'Klebsiella pneumoniae', 'Pseudomonas aeruginosa']

# Reuse the combined metadata (all years) we already have as `metadata` from earlier exploration,
# or rebuild it via load_metadata_with_years-style loading if not in memory
non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

for species in target_species:
    print(f"\n{'='*60}")
    print(f"SPECIES: {species}")
    print('='*60)

    species_df = metadata[metadata['species'] == species]
    print(f"Total isolates: {len(species_df)}")

    for ab in antibiotic_cols:
        col = species_df[ab]
        non_missing = col[(col.notna()) & (col != '-')]
        if len(non_missing) < 200:  # skip antibiotics with too little data to matter
            continue

        # collapse to binary using same rule as labels.py: any R or I -> 1, pure S -> 0
        def to_binary(v):
            if 'R' in v: return 1
            if 'I' in v: return 1
            if 'S' in v: return 0
            return None

        binary = non_missing.apply(to_binary).dropna()
        if len(binary) < 200:
            continue

        r_rate = binary.mean()
        majority = "R-majority" if r_rate > 0.5 else "S-majority"
        print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f} ({majority})")


SPECIES: Staphylococcus aureus
Total isolates: 6994
  Piperacillin-Tazobactam: n=3640, R-rate=0.196 (S-majority)
  Meropenem: n=3643, R-rate=0.195 (S-majority)
  Ciprofloxacin: n=3789, R-rate=0.171 (S-majority)
  Cefepime: n=3640, R-rate=0.196 (S-majority)
  Cotrimoxazole: n=3771, R-rate=0.045 (S-majority)
  Imipenem: n=3640, R-rate=0.196 (S-majority)
  Ceftriaxone: n=3640, R-rate=0.196 (S-majority)
  Clindamycin: n=3635, R-rate=0.159 (S-majority)
  Amoxicillin-Clavulanic acid: n=3640, R-rate=0.196 (S-majority)
  Vancomycin: n=3791, R-rate=0.000 (S-majority)
  Penicillin: n=3634, R-rate=0.741 (R-majority)
  Erythromycin: n=3640, R-rate=0.190 (S-majority)
  Tetracycline: n=3643, R-rate=0.084 (S-majority)
  Ampicillin-Amoxicillin: n=3637, R-rate=0.741 (R-majority)
  Linezolid: n=3639, R-rate=0.000 (S-majority)
  Teicoplanin: n=3631, R-rate=0.002 (S-majority)
  Tigecycline: n=3640, R-rate=0.000 (S-majority)
  Daptomycin: n=3783, R-rate=0.009 (S-majority)
  Gentamicin: n=3643, R-rate=0.03

In [13]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

pen = species_df['Penicillin']
amp = species_df['Ampicillin-Amoxicillin']

# only compare where both are non-missing
both_present = (pen.notna()) & (pen != '-') & (amp.notna()) & (amp != '-')
print(f"Isolates with both non-missing: {both_present.sum()}")

pen_valid = pen[both_present]
amp_valid = amp[both_present]

exact_match = (pen_valid == amp_valid).mean()
print(f"Exact string match rate: {exact_match:.4f}")

# also check after collapsing to binary (R/I->1, S->0), since exact string 
# match is stricter than binary match (e.g. 'R(1), S(1)' vs 'R' would differ 
# as strings but both collapse to 1)
def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

pen_binary = pen_valid.apply(to_binary)
amp_binary = amp_valid.apply(to_binary)

binary_match = (pen_binary == amp_binary).mean()
print(f"Binary (R/I=1, S=0) match rate: {binary_match:.4f}")

Isolates with both non-missing: 3634
Exact string match rate: 1.0000
Binary (R/I=1, S=0) match rate: 1.0000


In [14]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']
non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

print("Near-balanced labels (40-55% R) for S. aureus:\n")
for ab in antibiotic_cols:
    col = species_df[ab]
    non_missing = col[(col.notna()) & (col != '-')]
    if len(non_missing) < 200:
        continue
    binary = non_missing.apply(to_binary).dropna()
    if len(binary) < 200:
        continue
    r_rate = binary.mean()
    if 0.40 <= r_rate <= 0.55:
        print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f}")

print("\n(if nothing printed above, no near-balanced labels exist for S. aureus)")

Near-balanced labels (40-55% R) for S. aureus:


(if nothing printed above, no near-balanced labels exist for S. aureus)


In [15]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

candidates = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem', 
              'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 
              'Oxacillin', 'Cefazolin']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

print("Pairwise binary match rates among suspiciously-identical beta-lactams:\n")
for i in range(len(candidates)):
    for j in range(i+1, len(candidates)):
        a, b = candidates[i], candidates[j]
        col_a, col_b = species_df[a], species_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        if match_rate > 0.95:
            print(f"  {a} <-> {b}: n={both_present.sum()}, match={match_rate:.4f}  *** LIKELY DUPLICATE ***")

Pairwise binary match rates among suspiciously-identical beta-lactams:

  Piperacillin-Tazobactam <-> Meropenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefepime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Imipenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Ceftriaxone: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Amoxicillin-Clavulanic acid: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefuroxime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Oxacillin: n=3640, match=0.9975  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefazolin: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Cefepime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Imipenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Ceftriaxone: n=3640, match=1.0000  *** LIKE

In [16]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

cluster_reps = ['Oxacillin', 'Piperacillin-Tazobactam', 'Cefazolin']
for rep in cluster_reps:
    col_pen, col_rep = species_df['Penicillin'], species_df[rep]
    both_present = (col_pen.notna()) & (col_pen != '-') & (col_rep.notna()) & (col_rep != '-')
    bin_pen = col_pen[both_present].apply(to_binary)
    bin_rep = col_rep[both_present].apply(to_binary)
    match_rate = (bin_pen == bin_rep).mean()
    print(f"Penicillin <-> {rep}: n={both_present.sum()}, match={match_rate:.4f}")

Penicillin <-> Oxacillin: n=3634, match=0.4576
Penicillin <-> Piperacillin-Tazobactam: n=3634, match=0.4551
Penicillin <-> Cefazolin: n=3634, match=0.4551


In [17]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

col_clin, col_ery = species_df['Clindamycin'], species_df['Erythromycin']
both_present = (col_clin.notna()) & (col_clin != '-') & (col_ery.notna()) & (col_ery != '-')
bin_clin = col_clin[both_present].apply(to_binary)
bin_ery = col_ery[both_present].apply(to_binary)
match_rate = (bin_clin == bin_ery).mean()
print(f"Clindamycin <-> Erythromycin: n={both_present.sum()}, match={match_rate:.4f}")

Clindamycin <-> Erythromycin: n=3631, match=0.9317


In [18]:
cluster = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem', 
           'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 
           'Oxacillin', 'Cefazolin']

print("Non-missing counts per antibiotic in cluster:")
for ab in cluster:
    col = species_df[ab]
    n = ((col.notna()) & (col != '-')).sum()
    print(f"  {ab}: n={n}")

# check if the SAME isolates (by code) are tested across all 9
masks = []
for ab in cluster:
    col = species_df[ab]
    masks.append((col.notna()) & (col != '-'))

all_same_mask = masks[0]
for m in masks[1:]:
    all_same_mask = all_same_mask & (m == masks[0])  # do all masks match mask[0] exactly, per-row?
print(f"\nRows where ALL 9 antibiotics share the exact same missingness pattern as Piperacillin-Tazobactam: {all_same_mask.sum()} / {len(species_df)}")

# quantify Oxacillin's disagreements specifically - how many isolates, which years
col_pip, col_oxa = species_df['Piperacillin-Tazobactam'], species_df['Oxacillin']
both = (col_pip.notna()) & (col_pip != '-') & (col_oxa.notna()) & (col_oxa != '-')
bin_pip = col_pip[both].apply(to_binary)
bin_oxa = col_oxa[both].apply(to_binary)
disagree_mask = bin_pip != bin_oxa
print(f"\nPiperacillin-Tazobactam vs Oxacillin disagreements: {disagree_mask.sum()} isolates")
if disagree_mask.sum() > 0:
    disagree_years = species_df.loc[both][disagree_mask.values]['year_folder']
    print("Disagreements by year:")
    print(disagree_years.value_counts().sort_index())

Non-missing counts per antibiotic in cluster:
  Piperacillin-Tazobactam: n=3640
  Meropenem: n=3643
  Cefepime: n=3640
  Imipenem: n=3640
  Ceftriaxone: n=3640
  Amoxicillin-Clavulanic acid: n=3640
  Cefuroxime: n=3640
  Oxacillin: n=3791
  Cefazolin: n=3640

Rows where ALL 9 antibiotics share the exact same missingness pattern as Piperacillin-Tazobactam: 3640 / 6994

Piperacillin-Tazobactam vs Oxacillin disagreements: 9 isolates
Disagreements by year:
year_folder
2017    3
2018    6
Name: count, dtype: int64


In [19]:
ecoli_df = metadata[metadata['species'] == 'Escherichia coli']

# test the same beta-lactam cluster antibiotics in E. coli
ecoli_cluster_candidates = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem',
                              'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 'Cefazolin']

print("Pairwise match rates for the same antibiotic cluster, tested on E. coli:\n")
for i in range(len(ecoli_cluster_candidates)):
    for j in range(i+1, len(ecoli_cluster_candidates)):
        a, b = ecoli_cluster_candidates[i], ecoli_cluster_candidates[j]
        col_a, col_b = ecoli_df[a], ecoli_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        flag = "  *** HIGH MATCH ***" if match_rate > 0.95 else ""
        print(f"  {a} <-> {b}: n={both_present.sum()}, match={match_rate:.4f}{flag}")

Pairwise match rates for the same antibiotic cluster, tested on E. coli:

  Piperacillin-Tazobactam <-> Meropenem: n=4863, match=0.9149
  Piperacillin-Tazobactam <-> Cefepime: n=4870, match=0.8078
  Piperacillin-Tazobactam <-> Imipenem: n=4862, match=0.9157
  Piperacillin-Tazobactam <-> Ceftriaxone: n=4870, match=0.7768
  Piperacillin-Tazobactam <-> Amoxicillin-Clavulanic acid: n=4865, match=0.8183
  Meropenem <-> Cefepime: n=4934, match=0.8121
  Meropenem <-> Imipenem: n=4933, match=0.9982  *** HIGH MATCH ***
  Meropenem <-> Ceftriaxone: n=4934, match=0.7756
  Meropenem <-> Amoxicillin-Clavulanic acid: n=4927, match=0.7327
  Cefepime <-> Imipenem: n=4934, match=0.8121
  Cefepime <-> Ceftriaxone: n=4986, match=0.9531  *** HIGH MATCH ***
  Cefepime <-> Amoxicillin-Clavulanic acid: n=4977, match=0.7555
  Imipenem <-> Ceftriaxone: n=4934, match=0.7762
  Imipenem <-> Amoxicillin-Clavulanic acid: n=4927, match=0.7333
  Ceftriaxone <-> Amoxicillin-Clavulanic acid: n=4979, match=0.7499


In [20]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

# Check Penicillin's own availability first, since it's the non-negotiable seed
pen_col = species_df['Penicillin']
pen_available = (pen_col.notna()) & (pen_col != '-')
print(f"Penicillin availability: {pen_available.sum()} / {len(species_df)}")

core_four = ['Penicillin', 'Oxacillin', 'Ciprofloxacin', 'Cotrimoxazole']

panels = {
    'core4_plus_clindamycin': core_four + ['Clindamycin'],
    'core4_plus_erythromycin': core_four + ['Erythromycin'],
    'core4_plus_both': core_four + ['Clindamycin', 'Erythromycin'],
}

for panel_name, antibiotics in panels.items():
    mask = pd.Series(True, index=species_df.index)
    for ab in antibiotics:
        col = species_df[ab]
        mask &= (col.notna()) & (col != '-')
    print(f"\n{panel_name} ({len(antibiotics)} antibiotics): complete cases = {mask.sum()}")

    # per-year breakdown, since we'll need this for temporal split later
    year_counts = species_df.loc[mask, 'year_folder'].value_counts().sort_index()
    print(f"  Per-year: {dict(year_counts)}")

Penicillin availability: 3634 / 6994

core4_plus_clindamycin (5 antibiotics): complete cases = 3626
  Per-year: {'2015': np.int64(67), '2016': np.int64(1035), '2017': np.int64(1380), '2018': np.int64(1144)}

core4_plus_erythromycin (5 antibiotics): complete cases = 3632
  Per-year: {'2015': np.int64(67), '2016': np.int64(1036), '2017': np.int64(1381), '2018': np.int64(1148)}

core4_plus_both (6 antibiotics): complete cases = 3626
  Per-year: {'2015': np.int64(67), '2016': np.int64(1035), '2017': np.int64(1380), '2018': np.int64(1144)}


In [21]:
panel = ['Penicillin', 'Oxacillin', 'Ciprofloxacin', 'Cotrimoxazole', 'Clindamycin', 'Erythromycin']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

mask = pd.Series(True, index=species_df.index)
for ab in panel:
    col = species_df[ab]
    mask &= (col.notna()) & (col != '-')

complete_df = species_df[mask].copy()
print("Complete-case count:", len(complete_df))

labels_sa = complete_df[panel].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R) per antibiotic:")
print(labels_sa.mean().round(4))

print("\n6x6 correlation matrix:")
print(labels_sa.corr().round(3))

Complete-case count: 3626

Class balance (proportion R) per antibiotic:
Penicillin       0.7413
Oxacillin        0.1980
Ciprofloxacin    0.1616
Cotrimoxazole    0.0339
Clindamycin      0.1586
Erythromycin     0.1900
dtype: float64

6x6 correlation matrix:
               Penicillin  Oxacillin  Ciprofloxacin  Cotrimoxazole  \
Penicillin          1.000      0.294          0.145          0.086   
Oxacillin           0.294      1.000          0.353          0.148   
Ciprofloxacin       0.145      0.353          1.000          0.129   
Cotrimoxazole       0.086      0.148          0.129          1.000   
Clindamycin         0.067      0.192          0.168          0.044   
Erythromycin        0.082      0.248          0.234          0.049   

               Clindamycin  Erythromycin  
Penicillin           0.067         0.082  
Oxacillin            0.192         0.248  
Ciprofloxacin        0.168         0.234  
Cotrimoxazole        0.044         0.049  
Clindamycin          1.000         0.7

In [22]:
corr_matrix = labels_sa.corr().abs()
for ab in panel:
    mean_corr = corr_matrix[ab].drop(ab).mean()
    print(f"{ab}: mean |correlation| with others = {mean_corr:.4f}")

Penicillin: mean |correlation| with others = 0.1347
Oxacillin: mean |correlation| with others = 0.2468
Ciprofloxacin: mean |correlation| with others = 0.2059
Cotrimoxazole: mean |correlation| with others = 0.0911
Clindamycin: mean |correlation| with others = 0.2476
Erythromycin: mean |correlation| with others = 0.2762


In [6]:
import pandas as pd
from src.data.load import load_metadata_with_years
import glob, os
from src.config import ID_FOLDER

paths = glob.glob(os.path.join(ID_FOLDER, '*', '*_clean.csv'))
frames = []
for path in paths:
    year_folder = os.path.basename(os.path.dirname(path))
    df = pd.read_csv(path, low_memory=False)
    df['year_folder'] = year_folder
    frames.append(df)
metadata = pd.concat(frames, ignore_index=True)

species_df = metadata[metadata['species'] == 'Staphylococcus aureus']
print("species_df shape:", species_df.shape)

species_df shape: (6994, 93)


In [7]:
panel_5 = ['Penicillin', 'Erythromycin', 'Clindamycin', 'Oxacillin', 'Ciprofloxacin']

mask_5 = pd.Series(True, index=species_df.index)
for ab in panel_5:
    col = species_df[ab]
    mask_5 &= (col.notna()) & (col != '-')

print(f"Complete-case count (5-antibiotic panel, no Cotrimoxazole): {mask_5.sum()}")

complete_df_5 = species_df[mask_5].copy()
year_counts_5 = complete_df_5['year_folder'].value_counts().sort_index()
print(f"Per-year: {dict(year_counts_5)}")

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

labels_sa_5 = complete_df_5[panel_5].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R):")
print(labels_sa_5.mean().round(4))

print("\n5x5 correlation matrix:")
print(labels_sa_5.corr().round(3))

print("\nMean |correlation| with others (for control-seed decision):")
corr_matrix_5 = labels_sa_5.corr().abs()
for ab in panel_5:
    mean_corr = corr_matrix_5[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Complete-case count (5-antibiotic panel, no Cotrimoxazole): 3626
Per-year: {'2015': np.int64(67), '2016': np.int64(1035), '2017': np.int64(1380), '2018': np.int64(1144)}

Class balance (proportion R):
Penicillin       0.7413
Erythromycin     0.1900
Clindamycin      0.1586
Oxacillin        0.1980
Ciprofloxacin    0.1616
dtype: float64

5x5 correlation matrix:
               Penicillin  Erythromycin  Clindamycin  Oxacillin  Ciprofloxacin
Penicillin          1.000         0.082        0.067      0.294          0.145
Erythromycin        0.082         1.000        0.767      0.248          0.234
Clindamycin         0.067         0.767        1.000      0.192          0.168
Oxacillin           0.294         0.248        0.192      1.000          0.353
Ciprofloxacin       0.145         0.234        0.168      0.353          1.000

Mean |correlation| with others (for control-seed decision):
  Penicillin: 0.1468
  Erythromycin: 0.3330
  Clindamycin: 0.2985
  Oxacillin: 0.2716
  Ciprofloxacin: 0

In [8]:
from maldi_learn.driams import load_driams_dataset
from src.config_saureus import DATA_ROOT, SPECIES, ANTIBIOTICS
from src.data.split import temporal_split
from src.data.load import load_metadata_with_years
import numpy as np

dataset_sa = load_driams_dataset(
    root=str(DATA_ROOT.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES, antibiotics=ANTIBIOTICS,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
metadata_df = load_metadata_with_years()
split_sa = temporal_split(dataset_sa, metadata_df)

Y_sa = np.column_stack([dataset_sa.to_numpy(ab) for ab in ANTIBIOTICS])
Y_test_sa = Y_sa[split_sa['test_idx']]

true_all_zero_frac_sa = (Y_test_sa.sum(axis=1) == 0).mean()
print(f"True all-zero fraction (S. aureus temporal test set): {true_all_zero_frac_sa:.4f}")
print(f"n={len(Y_test_sa)}, all-zero count={int((Y_test_sa.sum(axis=1) == 0).sum())}")

temporal_split: train=2388, test=1066
True all-zero fraction (S. aureus temporal test set): 0.2195
n=1066, all-zero count=234


In [9]:
from maldi_learn.driams import load_driams_dataset
from src.config_saureus import DATA_ROOT, SPECIES, ANTIBIOTICS, RANDOM_SEED
from src.data.features import to_feature_matrix
from src.data.load import load_metadata_with_years
from src.data.split import temporal_split
from xgboost import XGBClassifier
import numpy as np

dataset_sa = load_driams_dataset(
    root=str(DATA_ROOT.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES, antibiotics=ANTIBIOTICS,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
X_sa = to_feature_matrix(dataset_sa)
metadata_df = load_metadata_with_years()
split_sa = temporal_split(dataset_sa, metadata_df)

train_idx, test_idx = split_sa['train_idx'], split_sa['test_idx']
X_train, X_test = X_sa[train_idx], X_sa[test_idx]

pred_matrix = np.zeros((len(test_idx), len(ANTIBIOTICS)), dtype=int)
for i, ab in enumerate(ANTIBIOTICS):
    y = dataset_sa.to_numpy(ab)
    y_train = y[train_idx]
    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train, y_train)
    pred_matrix[:, i] = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)

baseline_all_zero_frac = (pred_matrix.sum(axis=1) == 0).mean()
print(f"Baseline all_zero_frac: {baseline_all_zero_frac:.4f}")
print(f"True all_zero rate: 0.2195")

temporal_split: train=2388, test=1066
Baseline all_zero_frac: 0.1079
True all_zero rate: 0.2195


In [10]:
Y_test_sa = np.column_stack([dataset_sa.to_numpy(ab) for ab in ANTIBIOTICS])[test_idx]
pen_idx = ANTIBIOTICS.index('Penicillin')

# true modal pattern: Penicillin=1, all others=0
true_modal_mask = (Y_test_sa[:, pen_idx] == 1) & (Y_test_sa[:, [i for i in range(len(ANTIBIOTICS)) if i != pen_idx]].sum(axis=1) == 0)
print(f"True modal-outcome rate (Pen-R, rest S): {true_modal_mask.mean():.4f}")

# Now check each chain ordering's predicted modal-outcome rate
# (re-run chains quickly, or better: check against saved chains_saureus_*_temporal.csv 
# combined with the Y_pred matrices if we still have them in memory from the chains run)

True modal-outcome rate (Pen-R, rest S): 0.4578


In [11]:
# We already have baseline pred_matrix and dataset_sa/X_sa/split_sa from the previous check.
# Now let's get Option A's chain predictions the same way, cheaply, temporal-only.

from sklearn.multioutput import ClassifierChain

order_a = ANTIBIOTICS  # Option A uses ANTIBIOTICS' natural order (Penicillin first already)
Y_sa = np.column_stack([dataset_sa.to_numpy(ab) for ab in order_a])
Y_train_sa = Y_sa[train_idx]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order_a))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train_sa)
chain_pred_matrix = chain.predict(X_test)

# Check 1: agreement between baseline and chain predictions
agreement = (pred_matrix == chain_pred_matrix).all(axis=1).mean()
print(f"Fraction of test isolates with IDENTICAL 5-label prediction, baseline vs Option A chain: {agreement:.4f}")

# Check 2: per-label predicted positive rate, baseline vs chain vs true
Y_test_sa_full = Y_sa[test_idx]
print(f"\n{'Antibiotic':<15} {'True rate':<12} {'Baseline pred':<15} {'Chain pred':<12}")
for i, ab in enumerate(order_a):
    true_rate = Y_test_sa_full[:, i].mean()
    baseline_rate = pred_matrix[:, i].mean()
    chain_rate = chain_pred_matrix[:, i].mean()
    print(f"{ab:<15} {true_rate:<12.4f} {baseline_rate:<15.4f} {chain_rate:<12.4f}")

# Check 3: confirm modal outcome == majority-class pattern (all labels at their majority value)
majority_values = (Y_train_sa.mean(axis=0) > 0.5).astype(int)  # 1 if R-majority, 0 if S-majority, per label
print(f"\nMajority-class pattern per label (1=R-majority, 0=S-majority): {dict(zip(order_a, majority_values))}")
majority_pattern_mask = (Y_test_sa_full == majority_values).all(axis=1)
print(f"True rate of the exact majority-class pattern: {majority_pattern_mask.mean():.4f}")

Fraction of test isolates with IDENTICAL 5-label prediction, baseline vs Option A chain: 0.9268

Antibiotic      True rate    Baseline pred   Chain pred  
Penicillin      0.7167       0.8912          0.8912      
Erythromycin    0.1604       0.0113          0.0188      
Clindamycin     0.1332       0.0028          0.0141      
Oxacillin       0.1576       0.0713          0.0760      
Ciprofloxacin   0.1266       0.0507          0.0338      

Majority-class pattern per label (1=R-majority, 0=S-majority): {'Penicillin': np.int64(1), 'Erythromycin': np.int64(0), 'Clindamycin': np.int64(0), 'Oxacillin': np.int64(0), 'Ciprofloxacin': np.int64(0)}
True rate of the exact majority-class pattern: 0.4578


In [12]:
majority_values = np.array([1, 0, 0, 0, 0])  # Penicillin=1, rest=0, per the confirmed majority pattern

baseline_modal_rate = (pred_matrix == majority_values).all(axis=1).mean()
chain_modal_rate = (chain_pred_matrix == majority_values).all(axis=1).mean()
true_modal_rate = 0.4578  # already confirmed

print(f"True modal-outcome rate: {true_modal_rate:.4f}")
print(f"Baseline predicted modal-outcome rate: {baseline_modal_rate:.4f}")
print(f"Option A chain predicted modal-outcome rate: {chain_modal_rate:.4f}")

True modal-outcome rate: 0.4578
Baseline predicted modal-outcome rate: 0.7908
Option A chain predicted modal-outcome rate: 0.7992


In [13]:
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

for i, ab in enumerate(order_a):
    y_true = Y_test_sa_full[:, i]
    y_proba = chain.predict_proba(X_test)[:, i]  # or use a per-antibiotic baseline model's proba if preferred
    auroc = roc_auc_score(y_true, y_proba)

    # find the best possible F1 across all thresholds, to see how much headroom exists
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_f1 = f1_scores.max()
    best_threshold = thresholds[np.argmax(f1_scores[:-1])] if len(thresholds) > 0 else 0.5

    default_f1 = f1_score(y_true, (y_proba >= 0.5).astype(int))

    print(f"{ab}: AUROC={auroc:.4f}, F1@0.5={default_f1:.4f}, best_F1={best_f1:.4f} (at threshold={best_threshold:.4f})")

Penicillin: AUROC=0.7024, F1@0.5=0.8203, best_F1=0.8363 (at threshold=0.0598)
Erythromycin: AUROC=0.6234, F1@0.5=0.0838, best_F1=0.3351 (at threshold=0.0423)
Clindamycin: AUROC=0.5517, F1@0.5=0.1019, best_F1=0.2505 (at threshold=0.0002)
Oxacillin: AUROC=0.8278, F1@0.5=0.5783, best_F1=0.6204 (at threshold=0.2593)
Ciprofloxacin: AUROC=0.6837, F1@0.5=0.3977, best_F1=0.4490 (at threshold=0.2382)


In [14]:
kleb_df = metadata[metadata['species'] == 'Klebsiella pneumoniae']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

# Majority-class landscape (already partially known from earlier, recomputing cleanly)
print("Majority-class landscape for K. pneumoniae:\n")
usable_antibiotics = []
for ab in antibiotic_cols:
    col = kleb_df[ab]
    non_missing = col[(col.notna()) & (col != '-')]
    if len(non_missing) < 200:
        continue
    binary = non_missing.apply(to_binary).dropna()
    if len(binary) < 200:
        continue
    r_rate = binary.mean()
    majority = "R-majority" if r_rate > 0.5 else "S-majority"
    usable_antibiotics.append(ab)
    print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f} ({majority})")

# Duplicate check across ALL usable pairs (not just suspicious-looking ones this time,
# given what we found in S. aureus)
print("\nChecking all pairs for near-duplicates (>95% match):\n")
for i in range(len(usable_antibiotics)):
    for j in range(i+1, len(usable_antibiotics)):
        a, b = usable_antibiotics[i], usable_antibiotics[j]
        col_a, col_b = kleb_df[a], kleb_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        if match_rate > 0.95:
            print(f"  {a} <-> {b}: n={both_present.sum()}, match={match_rate:.4f}  *** LIKELY DUPLICATE ***")

print("\nDone checking duplicates.")

Majority-class landscape for K. pneumoniae:

  Piperacillin-Tazobactam: n=2787, R-rate=0.155 (S-majority)
  Meropenem: n=2855, R-rate=0.021 (S-majority)
  Ciprofloxacin: n=2864, R-rate=0.188 (S-majority)
  Cefepime: n=2864, R-rate=0.135 (S-majority)
  Cotrimoxazole: n=2862, R-rate=0.215 (S-majority)
  Ceftazidime: n=2857, R-rate=0.159 (S-majority)
  Amikacin: n=2857, R-rate=0.055 (S-majority)
  Levofloxacin: n=2856, R-rate=0.187 (S-majority)
  Imipenem: n=2860, R-rate=0.020 (S-majority)
  Tobramycin: n=2856, R-rate=0.115 (S-majority)
  Ceftriaxone: n=2864, R-rate=0.158 (S-majority)
  Colistin: n=2856, R-rate=0.005 (S-majority)
  Amoxicillin-Clavulanic acid: n=2852, R-rate=0.192 (S-majority)
  Ampicillin-Amoxicillin: n=2856, R-rate=1.000 (R-majority)
  Ertapenem: n=2868, R-rate=0.031 (S-majority)
  Cefpodoxime: n=1826, R-rate=0.129 (S-majority)
  Norfloxacin: n=1821, R-rate=0.160 (S-majority)
  Fosfomycin-Trometamol: n=1822, R-rate=0.260 (S-majority)

Checking all pairs for near-duplica

In [15]:
kleb_df = metadata[metadata['species'] == 'Klebsiella pneumoniae']

def availability_mask(ab):
    col = kleb_df[ab]
    return (col.notna()) & (col != '-')

carbapenem_cluster = ['Meropenem', 'Imipenem', 'Ertapenem', 'Colistin']
print("Availability counts:")
masks = {}
for ab in carbapenem_cluster:
    m = availability_mask(ab)
    masks[ab] = m
    print(f"  {ab}: n={m.sum()}")

print("\nPairwise missingness-pattern match (do they share the SAME tested/untested isolates?):")
for i in range(len(carbapenem_cluster)):
    for j in range(i+1, len(carbapenem_cluster)):
        a, b = carbapenem_cluster[i], carbapenem_cluster[j]
        same_pattern = (masks[a] == masks[b]).mean()
        print(f"  {a} <-> {b}: missingness pattern match = {same_pattern:.4f}")

# same check for the borderline pair
print("\nBorderline pair - Piperacillin-Tazobactam vs Amoxicillin-Clavulanic acid:")
m_pip = availability_mask('Piperacillin-Tazobactam')
m_amc = availability_mask('Amoxicillin-Clavulanic acid')
print(f"  Piperacillin-Tazobactam: n={m_pip.sum()}, Amoxicillin-Clavulanic acid: n={m_amc.sum()}")
print(f"  Missingness pattern match: {(m_pip == m_amc).mean():.4f}")

# reference point: what did the confirmed TRUE cluster (cephalosporins) look like?
print("\nReference - cephalosporin cluster (confirmed via S. aureus-style near-100% value match):")
cephalosporin_cluster = ['Cefepime', 'Ceftazidime', 'Ceftriaxone', 'Cefpodoxime']
for i in range(len(cephalosporin_cluster)):
    for j in range(i+1, len(cephalosporin_cluster)):
        a, b = cephalosporin_cluster[i], cephalosporin_cluster[j]
        ma, mb = availability_mask(a), availability_mask(b)
        same_pattern = (ma == mb).mean()
        print(f"  {a} <-> {b}: missingness pattern match = {same_pattern:.4f}")

# reference point: fluoroquinolones (also confirmed 100% value match, likely shared reporting)
print("\nReference - fluoroquinolone cluster:")
fq_cluster = ['Ciprofloxacin', 'Levofloxacin', 'Norfloxacin']
for i in range(len(fq_cluster)):
    for j in range(i+1, len(fq_cluster)):
        a, b = fq_cluster[i], fq_cluster[j]
        ma, mb = availability_mask(a), availability_mask(b)
        same_pattern = (ma == mb).mean()
        print(f"  {a} <-> {b}: missingness pattern match = {same_pattern:.4f}")

Availability counts:
  Meropenem: n=2855
  Imipenem: n=2860
  Ertapenem: n=2868
  Colistin: n=2856

Pairwise missingness-pattern match (do they share the SAME tested/untested isolates?):
  Meropenem <-> Imipenem: missingness pattern match = 0.9982
  Meropenem <-> Ertapenem: missingness pattern match = 0.9952
  Meropenem <-> Colistin: missingness pattern match = 0.9957
  Imipenem <-> Ertapenem: missingness pattern match = 0.9969
  Imipenem <-> Colistin: missingness pattern match = 0.9974
  Ertapenem <-> Colistin: missingness pattern match = 0.9944

Borderline pair - Piperacillin-Tazobactam vs Amoxicillin-Clavulanic acid:
  Piperacillin-Tazobactam: n=2787, Amoxicillin-Clavulanic acid: n=2852
  Missingness pattern match: 0.9814

Reference - cephalosporin cluster (confirmed via S. aureus-style near-100% value match):
  Cefepime <-> Ceftazidime: missingness pattern match = 0.9977
  Cefepime <-> Ceftriaxone: missingness pattern match = 1.0000
  Cefepime <-> Cefpodoxime: missingness pattern m

In [16]:
candidates = ['Ciprofloxacin', 'Ceftriaxone', 'Meropenem', 'Piperacillin-Tazobactam',
              'Cotrimoxazole', 'Amikacin', 'Tobramycin', 'Fosfomycin-Trometamol']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

mask_all8 = pd.Series(True, index=kleb_df.index)
for ab in candidates:
    col = kleb_df[ab]
    mask_all8 &= (col.notna()) & (col != '-')
print(f"Complete cases across all 8 candidates: {mask_all8.sum()}")

# per-antibiotic individual availability, for reference
for ab in candidates:
    col = kleb_df[ab]
    n = ((col.notna()) & (col != '-')).sum()
    print(f"  {ab}: individual n={n}")

complete_df_8 = kleb_df[mask_all8].copy()
labels_8 = complete_df_8[candidates].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R):")
print(labels_8.mean().round(4))

print("\n8x8 correlation matrix:")
print(labels_8.corr().round(3))

Complete cases across all 8 candidates: 1755
  Ciprofloxacin: individual n=2864
  Ceftriaxone: individual n=2864
  Meropenem: individual n=2855
  Piperacillin-Tazobactam: individual n=2787
  Cotrimoxazole: individual n=2862
  Amikacin: individual n=2857
  Tobramycin: individual n=2856
  Fosfomycin-Trometamol: individual n=1822

Class balance (proportion R):
Ciprofloxacin              0.1635
Ceftriaxone                0.1111
Meropenem                  0.0063
Piperacillin-Tazobactam    0.1168
Cotrimoxazole              0.1954
Amikacin                   0.0393
Tobramycin                 0.0900
Fosfomycin-Trometamol      0.2581
dtype: float64

8x8 correlation matrix:
                         Ciprofloxacin  Ceftriaxone  Meropenem  \
Ciprofloxacin                    1.000        0.623      0.180   
Ceftriaxone                      0.623        1.000      0.225   
Meropenem                        0.180        0.225      1.000   
Piperacillin-Tazobactam          0.573        0.645      0.218  

In [17]:
panel_options = {
    'with_amikacin': ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Amikacin'],
    'with_cotrimoxazole': ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole'],
    'both_no_tobramycin': ['Ceftriaxone', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Amikacin', 'Cotrimoxazole'],
}

for name, panel in panel_options.items():
    mask = pd.Series(True, index=kleb_df.index)
    for ab in panel:
        col = kleb_df[ab]
        mask &= (col.notna()) & (col != '-')
    print(f"\n{name} ({panel}): complete cases = {mask.sum()}")
    year_counts = kleb_df.loc[mask, 'year_folder'].value_counts().sort_index()
    print(f"  Per-year: {dict(year_counts)}")


with_amikacin (['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Amikacin']): complete cases = 2787
  Per-year: {'2015': np.int64(46), '2016': np.int64(746), '2017': np.int64(1235), '2018': np.int64(760)}

with_cotrimoxazole (['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']): complete cases = 2787
  Per-year: {'2015': np.int64(46), '2016': np.int64(746), '2017': np.int64(1235), '2018': np.int64(760)}

both_no_tobramycin (['Ceftriaxone', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Amikacin', 'Cotrimoxazole']): complete cases = 2787
  Per-year: {'2015': np.int64(46), '2016': np.int64(746), '2017': np.int64(1235), '2018': np.int64(760)}


In [18]:
panel_final = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']

mask_final = pd.Series(True, index=kleb_df.index)
for ab in panel_final:
    col = kleb_df[ab]
    mask_final &= (col.notna()) & (col != '-')

complete_df_final = kleb_df[mask_final].copy()
labels_final = complete_df_final[panel_final].apply(lambda col: col.apply(to_binary))

print("Class balance (proportion R):")
print(labels_final.mean().round(4))

print("\n5x5 correlation matrix:")
print(labels_final.corr().round(3))

print("\nMean |correlation| with others:")
corr_final = labels_final.corr().abs()
for ab in panel_final:
    mean_corr = corr_final[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Class balance (proportion R):
Ceftriaxone                0.1622
Tobramycin                 0.1180
Piperacillin-Tazobactam    0.1550
Ciprofloxacin              0.1905
Cotrimoxazole              0.2192
dtype: float64

5x5 correlation matrix:
                         Ceftriaxone  Tobramycin  Piperacillin-Tazobactam  \
Ceftriaxone                    1.000       0.648                    0.640   
Tobramycin                     0.648       1.000                    0.575   
Piperacillin-Tazobactam        0.640       0.575                    1.000   
Ciprofloxacin                  0.617       0.610                    0.534   
Cotrimoxazole                  0.571       0.543                    0.446   

                         Ciprofloxacin  Cotrimoxazole  
Ceftriaxone                      0.617          0.571  
Tobramycin                       0.610          0.543  
Piperacillin-Tazobactam          0.534          0.446  
Ciprofloxacin                    1.000          0.604  
Cotrimoxazole    

In [19]:
# Check Meropenem and Amikacin's correlation with the current panel specifically
extra_check = ['Meropenem', 'Amikacin']
mask_check = mask_final.copy()
for ab in extra_check:
    col = kleb_df[ab]
    mask_check &= (col.notna()) & (col != '-')

complete_df_check = kleb_df[mask_check].copy()
full_panel_check = panel_final + extra_check
labels_check = complete_df_check[full_panel_check].apply(lambda col: col.apply(to_binary))

print("Complete-case count with Meropenem+Amikacin added:", mask_check.sum())
print("\nMean |correlation| with the core 4 (Ceftriaxone, Tobramycin, Piperacillin-Tazobactam, Ciprofloxacin):")
core4 = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin']
corr_check = labels_check.corr().abs()
for ab in full_panel_check:
    if ab in core4:
        continue
    mean_corr_with_core = corr_check.loc[ab, core4].mean()
    print(f"  {ab}: mean |corr| with core 4 = {mean_corr_with_core:.4f}")

Complete-case count with Meropenem+Amikacin added: 2781

Mean |correlation| with the core 4 (Ceftriaxone, Tobramycin, Piperacillin-Tazobactam, Ciprofloxacin):
  Cotrimoxazole: mean |corr| with core 4 = 0.5403
  Meropenem: mean |corr| with core 4 = 0.2440
  Amikacin: mean |corr| with core 4 = 0.4883


In [20]:
panel_v2 = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Meropenem']

mask_v2 = pd.Series(True, index=kleb_df.index)
for ab in panel_v2:
    col = kleb_df[ab]
    mask_v2 &= (col.notna()) & (col != '-')

complete_df_v2 = kleb_df[mask_v2].copy()
labels_v2 = complete_df_v2[panel_v2].apply(lambda col: col.apply(to_binary))

print("Complete-case count:", mask_v2.sum())
year_counts_v2 = complete_df_v2['year_folder'].value_counts().sort_index()
print("Per-year:", dict(year_counts_v2))

print("\nClass balance (proportion R):")
print(labels_v2.mean().round(4))

print("\n5x5 correlation matrix:")
print(labels_v2.corr().round(3))

print("\nMean |correlation| with others:")
corr_v2 = labels_v2.corr().abs()
for ab in panel_v2:
    mean_corr = corr_v2[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Complete-case count: 2781
Per-year: {'2015': np.int64(46), '2016': np.int64(744), '2017': np.int64(1231), '2018': np.int64(760)}

Class balance (proportion R):
Ceftriaxone                0.1607
Tobramycin                 0.1176
Piperacillin-Tazobactam    0.1532
Ciprofloxacin              0.1895
Meropenem                  0.0209
dtype: float64

5x5 correlation matrix:
                         Ceftriaxone  Tobramycin  Piperacillin-Tazobactam  \
Ceftriaxone                    1.000       0.649                    0.638   
Tobramycin                     0.649       1.000                    0.576   
Piperacillin-Tazobactam        0.638       0.576                    1.000   
Ciprofloxacin                  0.618       0.610                    0.533   
Meropenem                      0.320       0.165                    0.343   

                         Ciprofloxacin  Meropenem  
Ceftriaxone                      0.618      0.320  
Tobramycin                       0.610      0.165  
Piperacilli

In [21]:
Y_kleb = labels_v2.values
true_all_zero_kleb = (Y_kleb.sum(axis=1) == 0).mean()
print(f"True all-zero rate (all 5 antibiotics, full complete-case set): {true_all_zero_kleb:.4f}")

majority_pattern_kleb = (labels_v2.mean() > 0.5).astype(int)
print(f"\nMajority-class pattern per label: {dict(majority_pattern_kleb)}")
# expect all 0s (S-majority) since no R-majority label exists in this panel

True all-zero rate (all 5 antibiotics, full complete-case set): 0.7307

Majority-class pattern per label: {'Ceftriaxone': np.int64(0), 'Tobramycin': np.int64(0), 'Piperacillin-Tazobactam': np.int64(0), 'Ciprofloxacin': np.int64(0), 'Meropenem': np.int64(0)}


In [24]:
from maldi_learn.driams import load_driams_dataset
from src.config_kpneumoniae import DATA_ROOT, SPECIES, ANTIBIOTICS, RANDOM_SEED
from src.data.features import to_feature_matrix
from src.data.load import load_metadata_with_years
from src.data.split import temporal_split
from xgboost import XGBClassifier
import numpy as np

dataset_kp = load_driams_dataset(
    root=str(DATA_ROOT.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES, antibiotics=ANTIBIOTICS,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
X_kp = to_feature_matrix(dataset_kp)
metadata_kp = load_metadata_with_years()
split_kp = temporal_split(dataset_kp, metadata_kp)

train_idx, test_idx = split_kp['train_idx'], split_kp['test_idx']
X_train, X_test = X_kp[train_idx], X_kp[test_idx]

true_matrix = np.zeros((len(test_idx), len(ANTIBIOTICS)), dtype=int)
pred_matrix = np.zeros((len(test_idx), len(ANTIBIOTICS)), dtype=int)

for i, ab in enumerate(ANTIBIOTICS):
    y = dataset_kp.to_numpy(ab)
    y_train, y_test = y[train_idx], y[test_idx]
    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train, y_train)
    y_pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
    true_matrix[:, i] = y_test
    pred_matrix[:, i] = y_pred

print("Rebuilt true_matrix and pred_matrix, shape:", true_matrix.shape)

temporal_split: train=1979, test=744
Rebuilt true_matrix and pred_matrix, shape: (744, 5)


In [25]:
from sklearn.metrics import jaccard_score, hamming_loss
import numpy as np

# Using the true_matrix/pred_matrix from the temporal baseline run
# (rebuild quickly if not in memory)
jaccard_default = jaccard_score(true_matrix, pred_matrix, average='samples', zero_division=0)
jaccard_zd1 = jaccard_score(true_matrix, pred_matrix, average='samples', zero_division=1)

print(f"K. pneumoniae baseline (temporal) Jaccard, zero_division=0: {jaccard_default:.4f}")
print(f"K. pneumoniae baseline (temporal) Jaccard, zero_division=1: {jaccard_zd1:.4f}")

true_all_zero_rate_test = (true_matrix.sum(axis=1) == 0).mean()
print(f"\nTrue all-zero rate in this test set: {true_all_zero_rate_test:.4f}")
print(f"Difference between the two Jaccard conventions: {jaccard_zd1 - jaccard_default:.4f}")

K. pneumoniae baseline (temporal) Jaccard, zero_division=0: 0.0226
K. pneumoniae baseline (temporal) Jaccard, zero_division=1: 0.7820

True all-zero rate in this test set: 0.7648
Difference between the two Jaccard conventions: 0.7594


In [26]:
trivial_pred = np.zeros_like(true_matrix)

trivial_hamming = hamming_loss(true_matrix, trivial_pred)
trivial_jaccard_zd0 = jaccard_score(true_matrix, trivial_pred, average='samples', zero_division=0)
trivial_jaccard_zd1 = jaccard_score(true_matrix, trivial_pred, average='samples', zero_division=1)

print(f"Trivial all-zero predictor - Hamming loss: {trivial_hamming:.4f}")
print(f"Trivial all-zero predictor - Jaccard (zero_division=0): {trivial_jaccard_zd0:.4f}")
print(f"Trivial all-zero predictor - Jaccard (zero_division=1): {trivial_jaccard_zd1:.4f}")

print(f"\nActual model - Hamming loss: {hamming_loss(true_matrix, pred_matrix):.4f}")
print(f"Actual model - Jaccard (zero_division=0): {jaccard_default:.4f}")

Trivial all-zero predictor - Hamming loss: 0.1019
Trivial all-zero predictor - Jaccard (zero_division=0): 0.0000
Trivial all-zero predictor - Jaccard (zero_division=1): 0.7648

Actual model - Hamming loss: 0.0933
Actual model - Jaccard (zero_division=0): 0.0226


In [27]:
# E. coli baseline, temporal split - rebuild quickly
from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split
from src.config import ANTIBIOTICS as ECOLI_ANTIBIOTICS, RANDOM_SEED as ECOLI_SEED
from xgboost import XGBClassifier

dataset_ec = load_dataset()
X_ec = to_feature_matrix(dataset_ec)
metadata_ec = load_metadata_with_years()
split_ec = temporal_split(dataset_ec, metadata_ec)

train_idx_ec, test_idx_ec = split_ec['train_idx'], split_ec['test_idx']
X_train_ec, X_test_ec = X_ec[train_idx_ec], X_ec[test_idx_ec]

true_matrix_ec = np.zeros((len(test_idx_ec), len(ECOLI_ANTIBIOTICS)), dtype=int)
pred_matrix_ec = np.zeros((len(test_idx_ec), len(ECOLI_ANTIBIOTICS)), dtype=int)

for i, ab in enumerate(ECOLI_ANTIBIOTICS):
    y = dataset_ec.to_numpy(ab)
    y_train, y_test = y[train_idx_ec], y[test_idx_ec]
    model = XGBClassifier(random_state=ECOLI_SEED, eval_metric='logloss')
    model.fit(X_train_ec, y_train)
    y_pred = (model.predict_proba(X_test_ec)[:, 1] >= 0.5).astype(int)
    true_matrix_ec[:, i] = y_test
    pred_matrix_ec[:, i] = y_pred

ec_jaccard_zd0 = jaccard_score(true_matrix_ec, pred_matrix_ec, average='samples', zero_division=0)
ec_jaccard_zd1 = jaccard_score(true_matrix_ec, pred_matrix_ec, average='samples', zero_division=1)

print(f"E. coli baseline (temporal) Jaccard, zero_division=0: {ec_jaccard_zd0:.4f}")
print(f"E. coli baseline (temporal) Jaccard, zero_division=1: {ec_jaccard_zd1:.4f}")

temporal_split: train=3331, test=1313
E. coli baseline (temporal) Jaccard, zero_division=0: 0.2323
E. coli baseline (temporal) Jaccard, zero_division=1: 0.4174


In [28]:
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import jaccard_score

OPTION_A_ORDER = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
                   'Ciprofloxacin', 'Cotrimoxazole']

Y_ec = np.column_stack([dataset_ec.to_numpy(ab) for ab in OPTION_A_ORDER])
Y_train_ec = Y_ec[train_idx_ec]

chain_ec = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(OPTION_A_ORDER))),
    random_state=RANDOM_SEED,
)
chain_ec.fit(X_train_ec, Y_train_ec)
chain_pred_ec = chain_ec.predict(X_test_ec)

Y_test_ec = Y_ec[test_idx_ec]

option_a_jaccard_zd0 = jaccard_score(Y_test_ec, chain_pred_ec, average='samples', zero_division=0)
option_a_jaccard_zd1 = jaccard_score(Y_test_ec, chain_pred_ec, average='samples', zero_division=1)

print(f"E. coli Option A (temporal) Jaccard, zero_division=0: {option_a_jaccard_zd0:.4f}")
print(f"E. coli Option A (temporal) Jaccard, zero_division=1: {option_a_jaccard_zd1:.4f}")

print(f"\nBaseline: zd0={ec_jaccard_zd0:.4f}, zd1={ec_jaccard_zd1:.4f}")
print(f"Option A: zd0={option_a_jaccard_zd0:.4f}, zd1={option_a_jaccard_zd1:.4f}")
print(f"\nGap under zd0: {option_a_jaccard_zd0 - ec_jaccard_zd0:.4f}")
print(f"Gap under zd1: {option_a_jaccard_zd1 - ec_jaccard_zd1:.4f}")

E. coli Option A (temporal) Jaccard, zero_division=0: 0.2453
E. coli Option A (temporal) Jaccard, zero_division=1: 0.4373

Baseline: zd0=0.2323, zd1=0.4174
Option A: zd0=0.2453, zd1=0.4373

Gap under zd0: 0.0130
Gap under zd1: 0.0199


In [29]:
import importlib
from src.evaluation import metrics
importlib.reload(metrics)
from src.evaluation.metrics import multilabel_metrics

kp_metrics = multilabel_metrics(true_matrix, pred_matrix)
print("K. pneumoniae baseline (temporal) - all metrics:")
for k, v in kp_metrics.items():
    print(f"  {k}: {v}")

# trivial predictor comparison
trivial_pred = np.zeros_like(true_matrix)
kp_trivial_metrics = multilabel_metrics(true_matrix, trivial_pred)
print("\nTrivial all-zero predictor - all metrics:")
for k, v in kp_trivial_metrics.items():
    print(f"  {k}: {v}")

K. pneumoniae baseline (temporal) - all metrics:
  hamming_loss: 0.09327956989247312
  jaccard_score: 0.022603046594982072
  jaccard_score_zd1: 0.7820116487455196
  exact_match_ratio: 0.7688172043010753
  restricted_jaccard: 0.0960952380952381

Trivial all-zero predictor - all metrics:
  hamming_loss: 0.10188172043010753
  jaccard_score: 0.0
  jaccard_score_zd1: 0.7647849462365591
  exact_match_ratio: 0.7647849462365591
  restricted_jaccard: 0.0


c:\Admin - Vaishali\Academics\VITV\Project_4_1\Implementation\amr_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Admin - Vaishali\Academics\VITV\Project_4_1\Implementation\amr_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [30]:
# E. coli - reuse true_matrix_ec, and both baseline pred_matrix_ec and chain_pred_ec from earlier
from src.evaluation.metrics import exact_match_ratio, restricted_jaccard

ec_baseline_emr = exact_match_ratio(true_matrix_ec, pred_matrix_ec)
ec_optiona_emr = exact_match_ratio(Y_test_ec, chain_pred_ec)
print(f"E. coli baseline exact_match_ratio: {ec_baseline_emr:.4f}")
print(f"E. coli Option A exact_match_ratio: {ec_optiona_emr:.4f}")

ec_baseline_rj = restricted_jaccard(true_matrix_ec, pred_matrix_ec)
ec_optiona_rj = restricted_jaccard(Y_test_ec, chain_pred_ec)
print(f"\nE. coli baseline restricted_jaccard: {ec_baseline_rj:.4f}")
print(f"E. coli Option A restricted_jaccard: {ec_optiona_rj:.4f}")

E. coli baseline exact_match_ratio: 0.2544
E. coli Option A exact_match_ratio: 0.2704

E. coli baseline restricted_jaccard: 0.3652
E. coli Option A restricted_jaccard: 0.3858


In [31]:
from maldi_learn.driams import load_driams_dataset
from src.config_saureus import DATA_ROOT as DATA_ROOT_SA, SPECIES as SPECIES_SA, ANTIBIOTICS as ANTIBIOTICS_SA, RANDOM_SEED as SEED_SA
from src.data.split import temporal_split as temporal_split_sa

dataset_sa2 = load_driams_dataset(
    root=str(DATA_ROOT_SA.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES_SA, antibiotics=ANTIBIOTICS_SA,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
X_sa2 = to_feature_matrix(dataset_sa2)
metadata_sa2 = load_metadata_with_years()
split_sa2 = temporal_split_sa(dataset_sa2, metadata_sa2)

train_idx_sa, test_idx_sa = split_sa2['train_idx'], split_sa2['test_idx']
X_train_sa, X_test_sa = X_sa2[train_idx_sa], X_sa2[test_idx_sa]

true_matrix_sa = np.zeros((len(test_idx_sa), len(ANTIBIOTICS_SA)), dtype=int)
pred_matrix_sa = np.zeros((len(test_idx_sa), len(ANTIBIOTICS_SA)), dtype=int)

for i, ab in enumerate(ANTIBIOTICS_SA):
    y = dataset_sa2.to_numpy(ab)
    y_train, y_test = y[train_idx_sa], y[test_idx_sa]
    model = XGBClassifier(random_state=SEED_SA, eval_metric='logloss')
    model.fit(X_train_sa, y_train)
    y_pred = (model.predict_proba(X_test_sa)[:, 1] >= 0.5).astype(int)
    true_matrix_sa[:, i] = y_test
    pred_matrix_sa[:, i] = y_pred

sa_emr = exact_match_ratio(true_matrix_sa, pred_matrix_sa)
sa_rj = restricted_jaccard(true_matrix_sa, pred_matrix_sa)
print(f"S. aureus baseline exact_match_ratio: {sa_emr:.4f}")
print(f"S. aureus baseline restricted_jaccard: {sa_rj:.4f}")

temporal_split: train=2388, test=1066
S. aureus baseline exact_match_ratio: 0.4906
S. aureus baseline restricted_jaccard: 0.6874


In [32]:
panel_v3_cotrim = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']
panel_v3_amikacin = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Amikacin']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

for name, panel in [('with_cotrimoxazole', panel_v3_cotrim), ('with_amikacin', panel_v3_amikacin)]:
    mask = pd.Series(True, index=kleb_df.index)
    for ab in panel:
        col = kleb_df[ab]
        mask &= (col.notna()) & (col != '-')
    print(f"\n{name}: complete cases = {mask.sum()}")

    complete_df = kleb_df[mask].copy()
    labels = complete_df[panel].apply(lambda col: col.apply(to_binary))
    print("Class balance (proportion R):")
    print(labels.mean().round(4))

    print("Mean |correlation| with the other 4:")
    corr = labels.corr().abs()
    for ab in panel:
        mean_corr = corr[ab].drop(ab).mean()
        print(f"  {ab}: {mean_corr:.4f}")


with_cotrimoxazole: complete cases = 2787
Class balance (proportion R):
Ceftriaxone                0.1622
Tobramycin                 0.1180
Piperacillin-Tazobactam    0.1550
Ciprofloxacin              0.1905
Cotrimoxazole              0.2192
dtype: float64
Mean |correlation| with the other 4:
  Ceftriaxone: 0.6190
  Tobramycin: 0.5936
  Piperacillin-Tazobactam: 0.5488
  Ciprofloxacin: 0.5913
  Cotrimoxazole: 0.5412

with_amikacin: complete cases = 2787
Class balance (proportion R):
Ceftriaxone                0.1622
Tobramycin                 0.1180
Piperacillin-Tazobactam    0.1550
Ciprofloxacin              0.1905
Amikacin                   0.0560
dtype: float64
Mean |correlation| with the other 4:
  Ceftriaxone: 0.5901
  Tobramycin: 0.6243
  Piperacillin-Tazobactam: 0.5481
  Ciprofloxacin: 0.5389
  Amikacin: 0.4899


In [33]:
# quick check: true all-zero rate for this corrected 5-antibiotic panel, temporal test set
from src.config_kpneumoniae import ANTIBIOTICS as KP_ANTIBIOTICS

Y_kp_test = np.column_stack([dataset_kp.to_numpy(ab) for ab in KP_ANTIBIOTICS])[test_idx]
# (assuming dataset_kp and test_idx are still in memory from the earlier rebuild -
#  if not, this needs re-running with the fresh load_driams_dataset call from 
#  baseline_kpneumoniae.py's __main__ block)

true_all_zero_kp_corrected = (Y_kp_test.sum(axis=1) == 0).mean()
print(f"True all-zero rate (5-antibiotic panel, temporal test): {true_all_zero_kp_corrected:.4f}")

True all-zero rate (5-antibiotic panel, temporal test): 0.7648


In [34]:
ec_agreement = (chain_pred_ec == pred_matrix_ec).all(axis=1).mean()
print(f"E. coli chain-baseline agreement (Option A, temporal): {ec_agreement:.4f}")

E. coli chain-baseline agreement (Option A, temporal): 0.3831


In [35]:
import shap

# Reuse dataset_kp, X_kp, split_kp, train_idx, test_idx already in memory
order_kp = ['Ceftriaxone', 'Piperacillin-Tazobactam', 'Tobramycin', 'Ciprofloxacin', 'Cotrimoxazole']  # Option A

Y_kp_full = np.column_stack([dataset_kp.to_numpy(ab) for ab in order_kp])
Y_train_kp = Y_kp_full[train_idx]

chain_kp = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order_kp))),
    random_state=RANDOM_SEED,
)
chain_kp.fit(X_train, Y_train_kp)

# Step 1 (Piperacillin-Tazobactam) - first step with a prior-prediction feature available
X_sub_kp = X_test[:500]  # subsample for speed
step0_pred = chain_kp.estimators_[0].predict(X_sub_kp).reshape(-1, 1)
step1_input = np.hstack([X_sub_kp, step0_pred])

explainer_kp = shap.TreeExplainer(chain_kp.estimators_[1])
shap_values_kp = explainer_kp.shap_values(step1_input)
if isinstance(shap_values_kp, list):
    shap_values_kp = shap_values_kp[1] if len(shap_values_kp) > 1 else shap_values_kp[0]

spectral_shap_kp = shap_values_kp[:, :6000]
prior_shap_kp = shap_values_kp[:, 6000:]

fraction_prior_kp = np.abs(prior_shap_kp).sum() / np.abs(shap_values_kp).sum()
print(f"K. pneumoniae, step 1 (Piperacillin-Tazobactam), fraction_prior_attribution: {fraction_prior_kp:.4f}")

# also check variance of the prior feature itself
print(f"Variance of step-0 prediction (prior feature): {step0_pred.var():.4f}")
print(f"Step-0 prediction positive rate: {step0_pred.mean():.4f}")

KeyError: 'Cotrimoxazole'

In [38]:
import importlib
from src import config_kpneumoniae
importlib.reload(config_kpneumoniae)

from src.config_kpneumoniae import DATA_ROOT as DATA_ROOT_KP, SPECIES as SPECIES_KP, ANTIBIOTICS as ANTIBIOTICS_KP, RANDOM_SEED as SEED_KP

print("Reloaded ANTIBIOTICS_KP:", ANTIBIOTICS_KP)

Reloaded ANTIBIOTICS_KP: ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']


In [39]:
from maldi_learn.driams import load_driams_dataset
from src.config_kpneumoniae import DATA_ROOT as DATA_ROOT_KP, SPECIES as SPECIES_KP, ANTIBIOTICS as ANTIBIOTICS_KP, RANDOM_SEED as SEED_KP
from src.data.features import to_feature_matrix
from src.data.split import temporal_split

dataset_kp2 = load_driams_dataset(
    root=str(DATA_ROOT_KP.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES_KP, antibiotics=ANTIBIOTICS_KP,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
X_kp2 = to_feature_matrix(dataset_kp2)
metadata_kp2 = load_metadata_with_years()
split_kp2 = temporal_split(dataset_kp2, metadata_kp2)

train_idx_kp2, test_idx_kp2 = split_kp2['train_idx'], split_kp2['test_idx']
X_train_kp2, X_test_kp2 = X_kp2[train_idx_kp2], X_kp2[test_idx_kp2]

print("dataset_kp2 antibiotics:", dataset_kp2.y.columns.tolist())
print("n_samples:", dataset_kp2.n_samples)

temporal_split: train=1984, test=743
dataset_kp2 antibiotics: ['Piperacillin-Tazobactam', 'species', 'laboratory_species', 'Ciprofloxacin', 'Cotrimoxazole', 'code', 'Tobramycin', 'Ceftriaxone']
n_samples: 2727


In [40]:
import shap
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

order_kp = ['Ceftriaxone', 'Piperacillin-Tazobactam', 'Tobramycin', 'Ciprofloxacin', 'Cotrimoxazole']  # Option A

Y_kp_full = np.column_stack([dataset_kp2.to_numpy(ab) for ab in order_kp])
Y_train_kp = Y_kp_full[train_idx_kp2]

chain_kp = ClassifierChain(
    estimator=XGBClassifier(random_state=SEED_KP, eval_metric='logloss'),
    order=list(range(len(order_kp))),
    random_state=SEED_KP,
)
chain_kp.fit(X_train_kp2, Y_train_kp)

X_sub_kp = X_test_kp2[:500]
step0_pred = chain_kp.estimators_[0].predict(X_sub_kp).reshape(-1, 1)
step1_input = np.hstack([X_sub_kp, step0_pred])

explainer_kp = shap.TreeExplainer(chain_kp.estimators_[1])
shap_values_kp = explainer_kp.shap_values(step1_input)
if isinstance(shap_values_kp, list):
    shap_values_kp = shap_values_kp[1] if len(shap_values_kp) > 1 else shap_values_kp[0]

spectral_shap_kp = shap_values_kp[:, :6000]
prior_shap_kp = shap_values_kp[:, 6000:]

fraction_prior_kp = np.abs(prior_shap_kp).sum() / np.abs(shap_values_kp).sum()
print(f"K. pneumoniae, step 1 (Piperacillin-Tazobactam), fraction_prior_attribution: {fraction_prior_kp:.4f}")
print(f"Variance of step-0 prediction (prior feature): {step0_pred.var():.4f}")
print(f"Step-0 prediction positive rate: {step0_pred.mean():.4f}")

K. pneumoniae, step 1 (Piperacillin-Tazobactam), fraction_prior_attribution: 0.1632
Variance of step-0 prediction (prior feature): 0.0328
Step-0 prediction positive rate: 0.0340


In [41]:
# Reuse existing baseline probability outputs, or rebuild quickly if needed
# E. coli baseline probabilities (temporal, Option A order for consistency)
ec_probas = []
for i, ab in enumerate(OPTION_A_ORDER):
    y = dataset_ec.to_numpy(ab)
    y_train = y[train_idx_ec]
    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_ec, y_train)
    proba = model.predict_proba(X_test_ec)[:, 1]
    ec_probas.append(proba)
ec_probas = np.column_stack(ec_probas)

frac_boundary_ec = ((ec_probas >= 0.3) & (ec_probas <= 0.7)).mean()
print(f"E. coli - fraction of (isolate, antibiotic) predictions in [0.3, 0.7]: {frac_boundary_ec:.4f}")

# K. pneumoniae baseline probabilities (temporal, Option A order)
kp_probas = []
for i, ab in enumerate(order_kp):
    y = dataset_kp2.to_numpy(ab)
    y_train = y[train_idx_kp2]
    model = XGBClassifier(random_state=SEED_KP, eval_metric='logloss')
    model.fit(X_train_kp2, y_train)
    proba = model.predict_proba(X_test_kp2)[:, 1]
    kp_probas.append(proba)
kp_probas = np.column_stack(kp_probas)

frac_boundary_kp = ((kp_probas >= 0.3) & (kp_probas <= 0.7)).mean()
print(f"K. pneumoniae - fraction of (isolate, antibiotic) predictions in [0.3, 0.7]: {frac_boundary_kp:.4f}")

E. coli - fraction of (isolate, antibiotic) predictions in [0.3, 0.7]: 0.1397
K. pneumoniae - fraction of (isolate, antibiotic) predictions in [0.3, 0.7]: 0.0202


In [42]:
# chain_kp already fitted from the earlier SHAP check
kp_chain_probas = chain_kp.predict_proba(X_test_kp2)

mean_abs_diff_kp = np.abs(kp_chain_probas - kp_probas).mean()
print(f"K. pneumoniae - mean absolute probability difference, chain vs baseline: {mean_abs_diff_kp:.4f}")

# per-antibiotic breakdown
for i, ab in enumerate(order_kp):
    diff = np.abs(kp_chain_probas[:, i] - kp_probas[:, i]).mean()
    print(f"  {ab}: mean |prob diff| = {diff:.4f}")

K. pneumoniae - mean absolute probability difference, chain vs baseline: 0.0369
  Ceftriaxone: mean |prob diff| = 0.0000
  Piperacillin-Tazobactam: mean |prob diff| = 0.0373
  Tobramycin: mean |prob diff| = 0.0268
  Ciprofloxacin: mean |prob diff| = 0.0523
  Cotrimoxazole: mean |prob diff| = 0.0682


In [43]:
ec_chain_probas = chain_ec.predict_proba(X_test_ec)

mean_abs_diff_ec = np.abs(ec_chain_probas - ec_probas).mean()
print(f"E. coli - mean absolute probability difference, chain vs baseline: {mean_abs_diff_ec:.4f}")

for i, ab in enumerate(OPTION_A_ORDER):
    diff = np.abs(ec_chain_probas[:, i] - ec_probas[:, i]).mean()
    print(f"  {ab}: mean |prob diff| = {diff:.4f}")

E. coli - mean absolute probability difference, chain vs baseline: 0.1114
  Ampicillin-Amoxicillin: mean |prob diff| = 0.0000
  Ceftriaxone: mean |prob diff| = 0.0932
  Amoxicillin-Clavulanic acid: mean |prob diff| = 0.1276
  Ciprofloxacin: mean |prob diff| = 0.1085
  Cotrimoxazole: mean |prob diff| = 0.2278


In [44]:
pa_df = metadata[metadata['species'] == 'Pseudomonas aeruginosa']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

def availability_mask_pa(ab):
    col = pa_df[ab]
    return (col.notna()) & (col != '-')

non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

print("Majority-class landscape for P. aeruginosa:\n")
usable_pa = []
for ab in antibiotic_cols:
    col = pa_df[ab]
    non_missing = col[(col.notna()) & (col != '-')]
    if len(non_missing) < 200:
        continue
    binary = non_missing.apply(to_binary).dropna()
    if len(binary) < 200:
        continue
    r_rate = binary.mean()
    majority = "R-majority" if r_rate > 0.5 else "S-majority"
    usable_pa.append(ab)
    print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f} ({majority})")

print(f"\n{len(usable_pa)} usable antibiotics total.")

print("\nChecking all pairs for value-agreement >95% (potential duplicates):\n")
duplicate_candidates = []
for i in range(len(usable_pa)):
    for j in range(i+1, len(usable_pa)):
        a, b = usable_pa[i], usable_pa[j]
        col_a, col_b = pa_df[a], pa_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        if match_rate > 0.95:
            ma, mb = availability_mask_pa(a), availability_mask_pa(b)
            missingness_match = (ma == mb).mean()
            tier = "CORE (>95% missingness)" if missingness_match > 0.95 else \
                   "SECONDARY (70-95% missingness)" if missingness_match >= 0.70 else \
                   "UNCLEAR (<70% missingness)"
            print(f"  {a} <-> {b}: value_match={match_rate:.4f}, missingness_match={missingness_match:.4f}  [{tier}]")
            duplicate_candidates.append((a, b, match_rate, missingness_match))

print(f"\nDone. {len(duplicate_candidates)} candidate duplicate pairs found.")

Majority-class landscape for P. aeruginosa:

  Piperacillin-Tazobactam: n=3260, R-rate=0.216 (S-majority)
  Meropenem: n=3267, R-rate=0.178 (S-majority)
  Ciprofloxacin: n=3267, R-rate=0.231 (S-majority)
  Cefepime: n=3164, R-rate=0.197 (S-majority)
  Ceftazidime: n=2551, R-rate=0.125 (S-majority)
  Amikacin: n=2450, R-rate=0.097 (S-majority)
  Levofloxacin: n=2446, R-rate=0.168 (S-majority)
  Imipenem: n=2456, R-rate=0.169 (S-majority)
  Tobramycin: n=3259, R-rate=0.083 (S-majority)
  Colistin: n=3261, R-rate=0.016 (S-majority)
  Aztreonam: n=777, R-rate=0.727 (R-majority)

11 usable antibiotics total.

Checking all pairs for value-agreement >95% (potential duplicates):

  Meropenem <-> Imipenem: value_match=0.9666, missingness_match=0.8329  [SECONDARY (70-95% missingness)]
  Ciprofloxacin <-> Levofloxacin: value_match=0.9980, missingness_match=0.8308  [SECONDARY (70-95% missingness)]

Done. 2 candidate duplicate pairs found.


In [45]:
core_candidates = ['Piperacillin-Tazobactam', 'Ciprofloxacin', 'Tobramycin', 'Ceftazidime']  # drop Meropenem (dup w/ Imipenem) and Levofloxacin (dup w/ Cipro) for now

mask_no_azt = pd.Series(True, index=pa_df.index)
for ab in core_candidates:
    col = pa_df[ab]
    mask_no_azt &= (col.notna()) & (col != '-')
print(f"Complete cases WITHOUT Aztreonam (4-antibiotic core): {mask_no_azt.sum()}")

mask_with_azt = mask_no_azt.copy()
col_azt = pa_df['Aztreonam']
mask_with_azt &= (col_azt.notna()) & (col_azt != '-')
print(f"Complete cases WITH Aztreonam added: {mask_with_azt.sum()}")

print(f"\nCost of including Aztreonam: {mask_no_azt.sum() - mask_with_azt.sum()} isolates lost "
      f"({100 * (mask_no_azt.sum() - mask_with_azt.sum()) / mask_no_azt.sum():.1f}%)")

Complete cases WITHOUT Aztreonam (4-antibiotic core): 2542
Complete cases WITH Aztreonam added: 167

Cost of including Aztreonam: 2375 isolates lost (93.4%)


In [46]:
# Resolve duplicates: keep Ciprofloxacin over Levofloxacin (matches E. coli precedent),
# keep Meropenem over Imipenem (arbitrary but consistent - broader-spectrum reference)
panel_candidates_pa = ['Piperacillin-Tazobactam', 'Ciprofloxacin', 'Tobramycin', 
                        'Ceftazidime', 'Meropenem', 'Amikacin', 'Colistin']

mask_pa = pd.Series(True, index=pa_df.index)
for ab in panel_candidates_pa:
    col = pa_df[ab]
    mask_pa &= (col.notna()) & (col != '-')
print(f"Complete cases across all 7 remaining candidates: {mask_pa.sum()}")

for ab in panel_candidates_pa:
    col = pa_df[ab]
    n = ((col.notna()) & (col != '-')).sum()
    print(f"  {ab}: individual n={n}")

complete_df_pa = pa_df[mask_pa].copy()
labels_pa = complete_df_pa[panel_candidates_pa].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R):")
print(labels_pa.mean().round(4))

print("\n7x7 correlation matrix:")
print(labels_pa.corr().round(3))

print("\nMean |correlation| with others:")
corr_pa = labels_pa.corr().abs()
for ab in panel_candidates_pa:
    mean_corr = corr_pa[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Complete cases across all 7 remaining candidates: 2443
  Piperacillin-Tazobactam: individual n=3260
  Ciprofloxacin: individual n=3267
  Tobramycin: individual n=3259
  Ceftazidime: individual n=2551
  Meropenem: individual n=3267
  Amikacin: individual n=2450
  Colistin: individual n=3261

Class balance (proportion R):
Piperacillin-Tazobactam    0.1772
Ciprofloxacin              0.1678
Tobramycin                 0.0319
Ceftazidime                0.1277
Meropenem                  0.1420
Amikacin                   0.0966
Colistin                   0.0074
dtype: float64

7x7 correlation matrix:
                         Piperacillin-Tazobactam  Ciprofloxacin  Tobramycin  \
Piperacillin-Tazobactam                    1.000          0.288       0.196   
Ciprofloxacin                              0.288          1.000       0.336   
Tobramycin                                 0.196          0.336       1.000   
Ceftazidime                                0.722          0.268       0.321   
Merop

In [47]:
col_pip, col_ctz = pa_df['Piperacillin-Tazobactam'], pa_df['Ceftazidime']
both_present = (col_pip.notna()) & (col_pip != '-') & (col_ctz.notna()) & (col_ctz != '-')
bin_pip = col_pip[both_present].apply(to_binary)
bin_ctz = col_ctz[both_present].apply(to_binary)
match_rate = (bin_pip == bin_ctz).mean()
print(f"Piperacillin-Tazobactam <-> Ceftazidime: value_match={match_rate:.4f}, n={both_present.sum()}")

ma = availability_mask_pa('Piperacillin-Tazobactam')
mb = availability_mask_pa('Ceftazidime')
print(f"Missingness match: {(ma == mb).mean():.4f}")

Piperacillin-Tazobactam <-> Ceftazidime: value_match=0.9214, n=2546
Missingness match: 0.8518


In [48]:
panel_pa_final = ['Piperacillin-Tazobactam', 'Ciprofloxacin', 'Tobramycin', 'Meropenem', 'Colistin']

mask_pa_final = pd.Series(True, index=pa_df.index)
for ab in panel_pa_final:
    col = pa_df[ab]
    mask_pa_final &= (col.notna()) & (col != '-')

complete_df_pa_final = pa_df[mask_pa_final].copy()
print(f"Complete-case count: {mask_pa_final.sum()}")

year_counts_pa = complete_df_pa_final['year_folder'].value_counts().sort_index()
print(f"Per-year: {dict(year_counts_pa)}")

labels_pa_final = complete_df_pa_final[panel_pa_final].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R):")
print(labels_pa_final.mean().round(4))

print("\n5x5 correlation matrix:")
print(labels_pa_final.corr().round(3))

print("\nMean |correlation| with others:")
corr_pa_final = labels_pa_final.corr().abs()
for ab in panel_pa_final:
    mean_corr = corr_pa_final[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Complete-case count: 3253
Per-year: {'2015': np.int64(82), '2016': np.int64(1002), '2017': np.int64(1219), '2018': np.int64(950)}

Class balance (proportion R):
Piperacillin-Tazobactam    0.2152
Ciprofloxacin              0.2299
Tobramycin                 0.0830
Meropenem                  0.1761
Colistin                   0.0157
dtype: float64

5x5 correlation matrix:
                         Piperacillin-Tazobactam  Ciprofloxacin  Tobramycin  \
Piperacillin-Tazobactam                    1.000          0.265       0.238   
Ciprofloxacin                              0.265          1.000       0.291   
Tobramycin                                 0.238          0.291       1.000   
Meropenem                                  0.427          0.357       0.200   
Colistin                                   0.078          0.031       0.106   

                         Meropenem  Colistin  
Piperacillin-Tazobactam      0.427     0.078  
Ciprofloxacin                0.357     0.031  
Tobramycin   

In [49]:
panel_no_colistin = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Tobramycin']

for replacement in ['Ceftazidime', 'Amikacin']:
    panel_test = panel_no_colistin + [replacement]
    mask = pd.Series(True, index=pa_df.index)
    for ab in panel_test:
        col = pa_df[ab]
        mask &= (col.notna()) & (col != '-')
    print(f"\nWith {replacement}: complete cases = {mask.sum()}")

    complete_df_test = pa_df[mask].copy()
    labels_test = complete_df_test[panel_test].apply(lambda col: col.apply(to_binary))
    print(f"  {replacement} R-rate: {labels_test[replacement].mean():.4f}")
    print(f"  Mean |corr| of {replacement} with core 4: {labels_test.corr().abs()[replacement].drop(replacement).mean():.4f}")


With Ceftazidime: complete cases = 2542
  Ceftazidime R-rate: 0.1255
  Mean |corr| of Ceftazidime with core 4: 0.4084

With Amikacin: complete cases = 2446
  Amikacin R-rate: 0.0969
  Mean |corr| of Amikacin with core 4: 0.2796


In [50]:
panel_pa_v2 = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Tobramycin', 'Amikacin']

mask_v2 = pd.Series(True, index=pa_df.index)
for ab in panel_pa_v2:
    col = pa_df[ab]
    mask_v2 &= (col.notna()) & (col != '-')

complete_df_v2 = pa_df[mask_v2].copy()
print(f"Complete-case count: {mask_v2.sum()}")

year_counts_v2 = complete_df_v2['year_folder'].value_counts().sort_index()
print(f"Per-year: {dict(year_counts_v2)}")

labels_v2 = complete_df_v2[panel_pa_v2].apply(lambda col: col.apply(to_binary))
print("\nClass balance (proportion R):")
print(labels_v2.mean().round(4))

print("\n5x5 correlation matrix:")
print(labels_v2.corr().round(3))

print("\nMean |correlation| with others:")
corr_v2 = labels_v2.corr().abs()
for ab in panel_pa_v2:
    mean_corr = corr_v2[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

Complete-case count: 2446
Per-year: {'2015': np.int64(65), '2016': np.int64(657), '2017': np.int64(954), '2018': np.int64(770)}

Class balance (proportion R):
Piperacillin-Tazobactam    0.1778
Meropenem                  0.1419
Ciprofloxacin              0.1684
Tobramycin                 0.0319
Amikacin                   0.0969
dtype: float64

5x5 correlation matrix:
                         Piperacillin-Tazobactam  Meropenem  Ciprofloxacin  \
Piperacillin-Tazobactam                    1.000      0.467          0.291   
Meropenem                                  0.467      1.000          0.346   
Ciprofloxacin                              0.291      0.346          1.000   
Tobramycin                                 0.196      0.200          0.335   
Amikacin                                   0.205      0.188          0.274   

                         Tobramycin  Amikacin  
Piperacillin-Tazobactam       0.196     0.205  
Meropenem                     0.200     0.188  
Ciprofloxacin     

In [51]:
from maldi_learn.driams import load_driams_dataset
from src.config_paeruginosa import DATA_ROOT, SPECIES, ANTIBIOTICS, RANDOM_SEED
from src.data.features import to_feature_matrix
from src.data.load import load_metadata_with_years
from src.data.split import temporal_split

dataset_pa = load_driams_dataset(
    root=str(DATA_ROOT.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SPECIES, antibiotics=ANTIBIOTICS,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
print("n_samples:", dataset_pa.n_samples)

metadata_pa = load_metadata_with_years()
split_pa = temporal_split(dataset_pa, metadata_pa)
train_idx_pa, test_idx_pa = split_pa['train_idx'], split_pa['test_idx']

for ab in ANTIBIOTICS:
    y = dataset_pa.to_numpy(ab)
    y_train, y_test = y[train_idx_pa], y[test_idx_pa]
    print(f"{ab}: train R-rate={y_train.mean():.4f} (n={len(y_train)}), test R-rate={y_test.mean():.4f} (n={len(y_test)})")

n_samples: 2152
temporal_split: train=1451, test=701
Piperacillin-Tazobactam: train R-rate=0.1082 (n=1451), test R-rate=0.1612 (n=701)
Meropenem: train R-rate=0.1089 (n=1451), test R-rate=0.0984 (n=701)
Ciprofloxacin: train R-rate=0.1275 (n=1451), test R-rate=0.1027 (n=701)
Tobramycin: train R-rate=0.0338 (n=1451), test R-rate=0.0029 (n=701)
Amikacin: train R-rate=0.0813 (n=1451), test R-rate=0.0385 (n=701)


In [52]:
from src.config_paeruginosa import DATA_ROOT as DR, SPECIES as SP, RANDOM_SEED as RS
ANTIBIOTICS_TEST = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Tobramycin', 'Ceftazidime']

dataset_pa_ceftaz = load_driams_dataset(
    root=str(DR.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SP, antibiotics=ANTIBIOTICS_TEST,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
print("n_samples:", dataset_pa_ceftaz.n_samples)

split_pa_ceftaz = temporal_split(dataset_pa_ceftaz, metadata_pa)
train_idx_c, test_idx_c = split_pa_ceftaz['train_idx'], split_pa_ceftaz['test_idx']

for ab in ANTIBIOTICS_TEST:
    y = dataset_pa_ceftaz.to_numpy(ab)
    y_train, y_test = y[train_idx_c], y[test_idx_c]
    print(f"{ab}: train R-rate={y_train.mean():.4f} (n={len(y_train)}), test R-rate={y_test.mean():.4f} (n={len(y_test)})")

n_samples: 2292
temporal_split: train=1573, test=719
Piperacillin-Tazobactam: train R-rate=0.1093 (n=1573), test R-rate=0.1572 (n=719)
Meropenem: train R-rate=0.1093 (n=1573), test R-rate=0.0960 (n=719)
Ciprofloxacin: train R-rate=0.1392 (n=1573), test R-rate=0.1168 (n=719)
Tobramycin: train R-rate=0.0445 (n=1573), test R-rate=0.0125 (n=719)
Ceftazidime: train R-rate=0.0712 (n=1573), test R-rate=0.1140 (n=719)


In [53]:
ANTIBIOTICS_4 = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Ceftazidime']

dataset_pa_4 = load_driams_dataset(
    root=str(DR.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SP, antibiotics=ANTIBIOTICS_4,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
print("n_samples:", dataset_pa_4.n_samples)

split_pa_4 = temporal_split(dataset_pa_4, metadata_pa)
train_idx_4, test_idx_4 = split_pa_4['train_idx'], split_pa_4['test_idx']

for ab in ANTIBIOTICS_4:
    y = dataset_pa_4.to_numpy(ab)
    y_train, y_test = y[train_idx_4], y[test_idx_4]
    print(f"{ab}: train R-rate={y_train.mean():.4f} (n={len(y_train)}), test R-rate={y_test.mean():.4f} (n={len(y_test)})")

n_samples: 2293
temporal_split: train=1574, test=719
Piperacillin-Tazobactam: train R-rate=0.1093 (n=1574), test R-rate=0.1572 (n=719)
Meropenem: train R-rate=0.1093 (n=1574), test R-rate=0.0960 (n=719)
Ciprofloxacin: train R-rate=0.1391 (n=1574), test R-rate=0.1168 (n=719)
Ceftazidime: train R-rate=0.0712 (n=1574), test R-rate=0.1140 (n=719)


In [54]:
ANTIBIOTICS_5_CEFEPIME = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Ceftazidime', 'Cefepime']

# first check it's not a hidden duplicate with anything already in the panel
for existing in ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Ceftazidime']:
    col_a, col_b = pa_df['Cefepime'], pa_df[existing]
    both = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
    bin_a = col_a[both].apply(to_binary)
    bin_b = col_b[both].apply(to_binary)
    match = (bin_a == bin_b).mean()
    ma = availability_mask_pa('Cefepime')
    mb = availability_mask_pa(existing)
    miss_match = (ma == mb).mean()
    print(f"Cefepime <-> {existing}: value_match={match:.4f}, missingness_match={miss_match:.4f}, n={both.sum()}")

dataset_pa_cefepime = load_driams_dataset(
    root=str(DR.parent), site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'], species=SP, antibiotics=ANTIBIOTICS_5_CEFEPIME,
    handle_missing_resistance_measurements='remove_if_any_missing', spectra_type='binned_6000',
)
print("\nn_samples:", dataset_pa_cefepime.n_samples)

split_pa_cef = temporal_split(dataset_pa_cefepime, metadata_pa)
train_idx_cef, test_idx_cef = split_pa_cef['train_idx'], split_pa_cef['test_idx']

for ab in ANTIBIOTICS_5_CEFEPIME:
    y = dataset_pa_cefepime.to_numpy(ab)
    y_train, y_test = y[train_idx_cef], y[test_idx_cef]
    print(f"{ab}: train R-rate={y_train.mean():.4f} (n={len(y_train)}), test R-rate={y_test.mean():.4f} (n={len(y_test)})")

Cefepime <-> Piperacillin-Tazobactam: value_match=0.8534, missingness_match=0.9777, n=3158
Cefepime <-> Meropenem: value_match=0.8226, missingness_match=0.9784, n=3163
Cefepime <-> Ciprofloxacin: value_match=0.7805, missingness_match=0.9779, n=3162
Cefepime <-> Ceftazidime: value_match=0.9213, missingness_match=0.8324, n=2451

n_samples: 2192
temporal_split: train=1491, test=701
Piperacillin-Tazobactam: train R-rate=0.1066 (n=1491), test R-rate=0.1526 (n=701)
Meropenem: train R-rate=0.1066 (n=1491), test R-rate=0.0927 (n=701)
Ciprofloxacin: train R-rate=0.1315 (n=1491), test R-rate=0.1041 (n=701)
Ceftazidime: train R-rate=0.0684 (n=1491), test R-rate=0.1113 (n=701)
Cefepime: train R-rate=0.0590 (n=1491), test R-rate=0.0713 (n=701)


In [55]:
Y_pa_final = np.column_stack([dataset_pa_cefepime.to_numpy(ab) for ab in ANTIBIOTICS_5_CEFEPIME])
labels_pa_final_df = pd.DataFrame(Y_pa_final, columns=ANTIBIOTICS_5_CEFEPIME)

print("5x5 correlation matrix:")
print(labels_pa_final_df.corr().round(3))

print("\nMean |correlation| with others:")
corr_final_pa = labels_pa_final_df.corr().abs()
for ab in ANTIBIOTICS_5_CEFEPIME:
    mean_corr = corr_final_pa[ab].drop(ab).mean()
    print(f"  {ab}: {mean_corr:.4f}")

5x5 correlation matrix:
                         Piperacillin-Tazobactam  Meropenem  Ciprofloxacin  \
Piperacillin-Tazobactam                    1.000      0.410          0.261   
Meropenem                                  0.410      1.000          0.328   
Ciprofloxacin                              0.261      0.328          1.000   
Ceftazidime                                0.667      0.322          0.182   
Cefepime                                   0.577      0.378          0.269   

                         Ceftazidime  Cefepime  
Piperacillin-Tazobactam        0.667     0.577  
Meropenem                      0.322     0.378  
Ciprofloxacin                  0.182     0.269  
Ceftazidime                    1.000     0.600  
Cefepime                       0.600     1.000  

Mean |correlation| with others:
  Piperacillin-Tazobactam: 0.4788
  Meropenem: 0.3593
  Ciprofloxacin: 0.2602
  Ceftazidime: 0.4426
  Cefepime: 0.4559


In [57]:
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
from src.data.features import to_feature_matrix

ANTIBIOTICS_PA = ['Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Ceftazidime', 'Cefepime']

X_pa = to_feature_matrix(dataset_pa_cefepime)
X_train_pa, X_test_pa = X_pa[train_idx_cef], X_pa[test_idx_cef]

print("Threshold analysis for P. aeruginosa baseline (temporal split):\n")
for ab in ANTIBIOTICS_PA:
    y = dataset_pa_cefepime.to_numpy(ab)
    y_train, y_test = y[train_idx_cef], y[test_idx_cef]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_pa, y_train)
    y_proba = model.predict_proba(X_test_pa)[:, 1]

    auroc = roc_auc_score(y_test, y_proba)
    default_f1 = f1_score(y_test, (y_proba >= 0.5).astype(int))

    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_f1 = f1_scores.max()
    best_threshold = thresholds[np.argmax(f1_scores[:-1])] if len(thresholds) > 0 else 0.5

    print(f"{ab}: AUROC={auroc:.4f}, F1@0.5={default_f1:.4f}, best_F1={best_f1:.4f} (at threshold={best_threshold:.4f})")

Threshold analysis for P. aeruginosa baseline (temporal split):

Piperacillin-Tazobactam: AUROC=0.5329, F1@0.5=0.0000, best_F1=0.2714 (at threshold=0.0092)
Meropenem: AUROC=0.5207, F1@0.5=0.0000, best_F1=0.1888 (at threshold=0.0033)
Ciprofloxacin: AUROC=0.6018, F1@0.5=0.0000, best_F1=0.2300 (at threshold=0.0090)
Ceftazidime: AUROC=0.5038, F1@0.5=0.0000, best_F1=0.2064 (at threshold=0.0007)
Cefepime: AUROC=0.5856, F1@0.5=0.0000, best_F1=0.1696 (at threshold=0.0014)


In [58]:
Y_pa_cc = np.column_stack([dataset_pa_cefepime.to_numpy(ab) for ab in ANTIBIOTICS_PA])

print("Full cohort vs. 5-antibiotic complete-case subset, resistance rates:\n")
for i, ab in enumerate(ANTIBIOTICS_PA):
    full_col = pa_df[ab]
    full_non_missing = full_col[(full_col.notna()) & (full_col != '-')]
    full_binary = full_non_missing.apply(to_binary).dropna()
    full_rate = full_binary.mean()

    cc_rate = Y_pa_cc[:, i].mean()
    print(f"{ab}: full-cohort R-rate={full_rate:.4f} (n={len(full_binary)})  |  complete-case R-rate={cc_rate:.4f} (n={len(Y_pa_cc)})")

Full cohort vs. 5-antibiotic complete-case subset, resistance rates:

Piperacillin-Tazobactam: full-cohort R-rate=0.2160 (n=3260)  |  complete-case R-rate=0.1214 (n=2192)
Meropenem: full-cohort R-rate=0.1785 (n=3267)  |  complete-case R-rate=0.1022 (n=2192)
Ciprofloxacin: full-cohort R-rate=0.2311 (n=3267)  |  complete-case R-rate=0.1227 (n=2192)
Ceftazidime: full-cohort R-rate=0.1254 (n=2551)  |  complete-case R-rate=0.0821 (n=2192)
Cefepime: full-cohort R-rate=0.1966 (n=3164)  |  complete-case R-rate=0.0630 (n=2192)


In [59]:
kleb_final_panel = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']

Y_kp_cc = np.column_stack([dataset_kp2.to_numpy(ab) for ab in kleb_final_panel])

print("K. pneumoniae: full cohort vs. complete-case resistance rates:\n")
for i, ab in enumerate(kleb_final_panel):
    full_col = kleb_df[ab]
    full_non_missing = full_col[(full_col.notna()) & (full_col != '-')]
    full_binary = full_non_missing.apply(to_binary).dropna()
    full_rate = full_binary.mean()

    cc_rate = Y_kp_cc[:, i].mean()
    print(f"{ab}: full-cohort R-rate={full_rate:.4f} (n={len(full_binary)})  |  complete-case R-rate={cc_rate:.4f} (n={len(Y_kp_cc)})")

K. pneumoniae: full cohort vs. complete-case resistance rates:

Ceftriaxone: full-cohort R-rate=0.1582 (n=2864)  |  complete-case R-rate=0.1522 (n=2727)
Tobramycin: full-cohort R-rate=0.1152 (n=2856)  |  complete-case R-rate=0.1093 (n=2727)
Piperacillin-Tazobactam: full-cohort R-rate=0.1550 (n=2787)  |  complete-case R-rate=0.1456 (n=2727)
Ciprofloxacin: full-cohort R-rate=0.1882 (n=2864)  |  complete-case R-rate=0.1801 (n=2727)
Cotrimoxazole: full-cohort R-rate=0.2145 (n=2862)  |  complete-case R-rate=0.2094 (n=2727)


In [60]:
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

print("E. coli temporal threshold-tuning demonstration:\n")
print(f"{'Antibiotic':<30} {'AUROC':<8} {'F1@0.5':<10} {'F1@tuned':<10} {'Tuned thresh':<12} {'Recovery'}")

for ab in ECOLI_ANTIBIOTICS:  # or OPTION_A_ORDER, whichever antibiotic list you want to report on
    y = dataset_ec.to_numpy(ab)
    y_train, y_test = y[train_idx_ec], y[test_idx_ec]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_ec, y_train)

    # Tune threshold using TRAINING predictions only (in-sample)
    train_proba = model.predict_proba(X_train_ec)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_train, train_proba)
    f1_scores_train = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_idx = np.argmax(f1_scores_train[:-1]) if len(thresholds) > 0 else None
    tuned_threshold = thresholds[best_idx] if best_idx is not None else 0.5

    # Apply to TEST (2018) predictions
    test_proba = model.predict_proba(X_test_ec)[:, 1]
    auroc = roc_auc_score(y_test, test_proba)
    f1_default = f1_score(y_test, (test_proba >= 0.5).astype(int))
    f1_tuned = f1_score(y_test, (test_proba >= tuned_threshold).astype(int))

    recovery = f1_tuned - f1_default
    print(f"{ab:<30} {auroc:<8.4f} {f1_default:<10.4f} {f1_tuned:<10.4f} {tuned_threshold:<12.4f} {recovery:+.4f}")

E. coli temporal threshold-tuning demonstration:

Antibiotic                     AUROC    F1@0.5     F1@tuned   Tuned thresh Recovery
Ciprofloxacin                  0.7945   0.4846     0.2315     0.9781       -0.2531
Cotrimoxazole                  0.6656   0.3261     0.0000     0.9722       -0.3261
Ceftriaxone                    0.8617   0.5539     0.1933     0.9798       -0.3606
Amoxicillin-Clavulanic acid    0.6042   0.1029     0.0000     0.9793       -0.1029
Ampicillin-Amoxicillin         0.6517   0.6764     0.2085     0.9775       -0.4679


In [61]:
# Build year-based masks for E. coli
years_ec = metadata_ec.loc[metadata_ec['code'].isin(dataset_ec.y['code']), ['code', 'year_folder']]
# Align with dataset_ec's row order
year_lookup_ec = dataset_ec.y[['code']].merge(years_ec, on='code', how='left')['year_folder'].astype(int).values

train_1516_mask = np.isin(year_lookup_ec, [2015, 2016])
train_17_mask = year_lookup_ec == 2017
test_18_mask = year_lookup_ec == 2018

print(f"2015-2016 (reduced train): {train_1516_mask.sum()}")
print(f"2017 (threshold-tuning holdout): {train_17_mask.sum()}")
print(f"2018 (final test): {test_18_mask.sum()}")

X_train_1516 = X_ec[train_1516_mask]
X_2017 = X_ec[train_17_mask]
X_test_18 = X_ec[test_18_mask]

print("\nBaseline AUROC comparison: full (2015-2017) train vs. reduced (2015-2016) train, evaluated on 2018:\n")
for ab in ECOLI_ANTIBIOTICS:
    y_full = dataset_ec.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask]
    y_test_18 = y_full[test_18_mask]

    # reduced model (2015-2016 only)
    model_reduced = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model_reduced.fit(X_train_1516, y_train_1516)
    proba_reduced = model_reduced.predict_proba(X_test_18)[:, 1]
    auroc_reduced = roc_auc_score(y_test_18, proba_reduced)

    # full model (2015-2017), already trained earlier as `model` variable per antibiotic - retrain for clarity
    model_full = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model_full.fit(X_train_ec, y_full[train_idx_ec])
    proba_full = model_full.predict_proba(X_test_ec)[:, 1]
    auroc_full = roc_auc_score(y_full[test_idx_ec], proba_full)

    print(f"{ab}: AUROC (2015-16 train) = {auroc_reduced:.4f}  |  AUROC (2015-17 train) = {auroc_full:.4f}  |  drop = {auroc_full - auroc_reduced:+.4f}")

2015-2016 (reduced train): 1379
2017 (threshold-tuning holdout): 1952
2018 (final test): 1313

Baseline AUROC comparison: full (2015-2017) train vs. reduced (2015-2016) train, evaluated on 2018:

Ciprofloxacin: AUROC (2015-16 train) = 0.7243  |  AUROC (2015-17 train) = 0.7945  |  drop = +0.0703
Cotrimoxazole: AUROC (2015-16 train) = 0.6169  |  AUROC (2015-17 train) = 0.6656  |  drop = +0.0487
Ceftriaxone: AUROC (2015-16 train) = 0.7951  |  AUROC (2015-17 train) = 0.8617  |  drop = +0.0665
Amoxicillin-Clavulanic acid: AUROC (2015-16 train) = 0.5641  |  AUROC (2015-17 train) = 0.6042  |  drop = +0.0401
Ampicillin-Amoxicillin: AUROC (2015-16 train) = 0.5977  |  AUROC (2015-17 train) = 0.6517  |  drop = +0.0540


In [62]:
print("E. coli threshold-tuning demonstration (train=2015-2016, tune=2017, test=2018):\n")
print(f"{'Antibiotic':<30} {'AUROC(test)':<12} {'F1@0.5':<10} {'F1@tuned':<10} {'Tuned thresh':<12} {'Recovery'}")

for ab in ECOLI_ANTIBIOTICS:
    y_full = dataset_ec.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask]
    y_2017 = y_full[train_17_mask]
    y_test_18 = y_full[test_18_mask]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_1516, y_train_1516)

    # Tune threshold on 2017 (held out from training, but temporally BEFORE the 2018 test set)
    proba_2017 = model.predict_proba(X_2017)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_2017, proba_2017)
    f1_scores_2017 = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_idx = np.argmax(f1_scores_2017[:-1]) if len(thresholds) > 0 else None
    tuned_threshold = thresholds[best_idx] if best_idx is not None else 0.5

    # Apply to 2018 test set
    proba_2018 = model.predict_proba(X_test_18)[:, 1]
    auroc_2018 = roc_auc_score(y_test_18, proba_2018)
    f1_default = f1_score(y_test_18, (proba_2018 >= 0.5).astype(int))
    f1_tuned = f1_score(y_test_18, (proba_2018 >= tuned_threshold).astype(int))

    recovery = f1_tuned - f1_default
    print(f"{ab:<30} {auroc_2018:<12.4f} {f1_default:<10.4f} {f1_tuned:<10.4f} {tuned_threshold:<12.4f} {recovery:+.4f}")

E. coli threshold-tuning demonstration (train=2015-2016, tune=2017, test=2018):

Antibiotic                     AUROC(test)  F1@0.5     F1@tuned   Tuned thresh Recovery
Ciprofloxacin                  0.7243       0.3783     0.5016     0.0751       +0.1233
Cotrimoxazole                  0.6169       0.2811     0.5100     0.0758       +0.2289
Ceftriaxone                    0.7951       0.3051     0.5263     0.0469       +0.2212
Amoxicillin-Clavulanic acid    0.5641       0.0311     0.4432     0.0030       +0.4121
Ampicillin-Amoxicillin         0.5977       0.7126     0.7417     0.0147       +0.0291


In [63]:
from sklearn.metrics import precision_score, recall_score

print("E. coli threshold-tuning: FULL comparison (same 2015-2016 model, both thresholds)\n")
print(f"{'Antibiotic':<30} {'AUROC':<8} {'F1@0.5':<8} {'F1@tuned':<9} {'Prec@0.5':<9} {'Rec@0.5':<8} {'Prec@tuned':<11} {'Rec@tuned':<10} {'TrueRate':<9} {'PredRate@0.5':<13} {'PredRate@tuned'}")

for ab in ECOLI_ANTIBIOTICS:
    y_full = dataset_ec.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask]
    y_2017 = y_full[train_17_mask]
    y_test_18 = y_full[test_18_mask]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_1516, y_train_1516)

    proba_2017 = model.predict_proba(X_2017)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_2017, proba_2017)
    f1_scores_2017 = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_idx = np.argmax(f1_scores_2017[:-1]) if len(thresholds) > 0 else None
    tuned_threshold = thresholds[best_idx] if best_idx is not None else 0.5

    proba_2018 = model.predict_proba(X_test_18)[:, 1]
    pred_05 = (proba_2018 >= 0.5).astype(int)
    pred_tuned = (proba_2018 >= tuned_threshold).astype(int)

    auroc = roc_auc_score(y_test_18, proba_2018)
    f1_05 = f1_score(y_test_18, pred_05)
    f1_tuned = f1_score(y_test_18, pred_tuned)
    prec_05 = precision_score(y_test_18, pred_05, zero_division=0)
    rec_05 = recall_score(y_test_18, pred_05, zero_division=0)
    prec_tuned = precision_score(y_test_18, pred_tuned, zero_division=0)
    rec_tuned = recall_score(y_test_18, pred_tuned, zero_division=0)

    true_rate = y_test_18.mean()
    pred_rate_05 = pred_05.mean()
    pred_rate_tuned = pred_tuned.mean()

    print(f"{ab:<30} {auroc:<8.4f} {f1_05:<8.4f} {f1_tuned:<9.4f} {prec_05:<9.4f} {rec_05:<8.4f} {prec_tuned:<11.4f} {rec_tuned:<10.4f} {true_rate:<9.4f} {pred_rate_05:<13.4f} {pred_rate_tuned:.4f}")

E. coli threshold-tuning: FULL comparison (same 2015-2016 model, both thresholds)

Antibiotic                     AUROC    F1@0.5   F1@tuned  Prec@0.5  Rec@0.5  Prec@tuned  Rec@tuned  TrueRate  PredRate@0.5  PredRate@tuned
Ciprofloxacin                  0.7243   0.3783   0.5016    0.6763    0.2626   0.4070      0.6536     0.2727    0.1059        0.4379
Cotrimoxazole                  0.6169   0.2811   0.5100    0.4265    0.2096   0.3737      0.8024     0.3161    0.1554        0.6786
Ceftriaxone                    0.7951   0.3051   0.5263    0.8182    0.1875   0.4545      0.6250     0.1828    0.0419        0.2513
Amoxicillin-Clavulanic acid    0.5641   0.0311   0.4432    0.5000    0.0160   0.2896      0.9439     0.2848    0.0091        0.9284
Ampicillin-Amoxicillin         0.5977   0.7126   0.7417    0.6291    0.8217   0.5895      1.0000     0.5895    0.7700        1.0000


In [64]:
y_full_aa = dataset_ec.to_numpy('Ampicillin-Amoxicillin')
y_train_1516_aa = y_full_aa[train_1516_mask]
y_2017_aa = y_full_aa[train_17_mask]
y_test_18_aa = y_full_aa[test_18_mask]

model_aa = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
model_aa.fit(X_train_1516, y_train_1516_aa)

proba_2017_aa = model_aa.predict_proba(X_2017)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_2017_aa, proba_2017_aa)
f1_scores_aa = 2 * precisions * recalls / (precisions + recalls + 1e-10)
best_idx_aa = np.argmax(f1_scores_aa[:-1])
tuned_threshold_aa = thresholds[best_idx_aa]

proba_2018_aa = model_aa.predict_proba(X_test_18)[:, 1]
pred_tuned_aa = (proba_2018_aa >= tuned_threshold_aa).astype(int)

print(f"Tuned threshold: {tuned_threshold_aa:.4f}")
print(f"True rate: {y_test_18_aa.mean():.4f}")
print(f"Predicted positive rate @ tuned threshold: {pred_tuned_aa.mean():.4f}")

Tuned threshold: 0.0147
True rate: 0.5895
Predicted positive rate @ tuned threshold: 1.0000


In [65]:
print("E. coli calibration-tuned threshold demonstration (train=2015-2016, calibrate=2017, test=2018):\n")
print(f"{'Antibiotic':<30} {'CalibThresh':<12} {'F1@calib':<9} {'Prec@calib':<11} {'Rec@calib':<10} {'TrueRate':<9} {'PredRate@calib'}")

for ab in ECOLI_ANTIBIOTICS:
    y_full = dataset_ec.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask]
    y_2017 = y_full[train_17_mask]
    y_test_18 = y_full[test_18_mask]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_1516, y_train_1516)

    proba_2017 = model.predict_proba(X_2017)[:, 1]
    observed_rate_2017 = y_2017.mean()

    # find threshold on 2017 where predicted positive rate matches observed rate
    candidate_thresholds = np.sort(proba_2017)[::-1]
    best_thresh = 0.5
    best_diff = float('inf')
    for t in np.unique(proba_2017):
        pred_rate = (proba_2017 >= t).mean()
        diff = abs(pred_rate - observed_rate_2017)
        if diff < best_diff:
            best_diff = diff
            best_thresh = t

    proba_2018 = model.predict_proba(X_test_18)[:, 1]
    pred_calib = (proba_2018 >= best_thresh).astype(int)

    f1_calib = f1_score(y_test_18, pred_calib)
    prec_calib = precision_score(y_test_18, pred_calib, zero_division=0)
    rec_calib = recall_score(y_test_18, pred_calib, zero_division=0)
    true_rate = y_test_18.mean()
    pred_rate_calib = pred_calib.mean()

    print(f"{ab:<30} {best_thresh:<12.4f} {f1_calib:<9.4f} {prec_calib:<11.4f} {rec_calib:<10.4f} {true_rate:<9.4f} {pred_rate_calib:.4f}")

E. coli calibration-tuned threshold demonstration (train=2015-2016, calibrate=2017, test=2018):

Antibiotic                     CalibThresh  F1@calib  Prec@calib  Rec@calib  TrueRate  PredRate@calib
Ciprofloxacin                  0.1497       0.5000    0.4843      0.5168     0.2727    0.2909
Cotrimoxazole                  0.2355       0.4328    0.3960      0.4771     0.3161    0.3808
Ceftriaxone                    0.0525       0.5360    0.4817      0.6042     0.1828    0.2292
Amoxicillin-Clavulanic acid    0.0817       0.3058    0.3670      0.2620     0.2848    0.2034
Ampicillin-Amoxicillin         0.6075       0.6876    0.6353      0.7494     0.5895    0.6954


In [66]:
print("E. coli calibration-tuned threshold — complete picture:\n")
print(f"{'Antibiotic':<30} {'AUROC':<8} {'CalibThresh':<12} {'F1@calib':<9} {'Prec@calib':<11} {'Rec@calib':<10} {'TrueRate':<9} {'PredRate@calib'}")

calib_results = {}
for ab in ECOLI_ANTIBIOTICS:
    y_full = dataset_ec.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask]
    y_2017 = y_full[train_17_mask]
    y_test_18 = y_full[test_18_mask]

    model = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss')
    model.fit(X_train_1516, y_train_1516)

    proba_2017 = model.predict_proba(X_2017)[:, 1]
    observed_rate_2017 = y_2017.mean()

    best_thresh = 0.5
    best_diff = float('inf')
    for t in np.unique(proba_2017):
        pred_rate = (proba_2017 >= t).mean()
        diff = abs(pred_rate - observed_rate_2017)
        if diff < best_diff:
            best_diff = diff
            best_thresh = t

    proba_2018 = model.predict_proba(X_test_18)[:, 1]
    pred_calib = (proba_2018 >= best_thresh).astype(int)

    auroc = roc_auc_score(y_test_18, proba_2018)
    f1_calib = f1_score(y_test_18, pred_calib)
    prec_calib = precision_score(y_test_18, pred_calib, zero_division=0)
    rec_calib = recall_score(y_test_18, pred_calib, zero_division=0)
    true_rate = y_test_18.mean()
    pred_rate_calib = pred_calib.mean()

    calib_results[ab] = {
        'auroc': auroc, 'f1': f1_calib, 'precision': prec_calib, 'recall': rec_calib,
        'true_rate': true_rate, 'pred_rate': pred_rate_calib, 'threshold': best_thresh
    }

    print(f"{ab:<30} {auroc:<8.4f} {best_thresh:<12.4f} {f1_calib:<9.4f} {prec_calib:<11.4f} {rec_calib:<10.4f} {true_rate:<9.4f} {pred_rate_calib:.4f}")

E. coli calibration-tuned threshold — complete picture:

Antibiotic                     AUROC    CalibThresh  F1@calib  Prec@calib  Rec@calib  TrueRate  PredRate@calib
Ciprofloxacin                  0.7243   0.1497       0.5000    0.4843      0.5168     0.2727    0.2909
Cotrimoxazole                  0.6169   0.2355       0.4328    0.3960      0.4771     0.3161    0.3808
Ceftriaxone                    0.7951   0.0525       0.5360    0.4817      0.6042     0.1828    0.2292
Amoxicillin-Clavulanic acid    0.5641   0.0817       0.3058    0.3670      0.2620     0.2848    0.2034
Ampicillin-Amoxicillin         0.5977   0.6075       0.6876    0.6353      0.7494     0.5895    0.6954


In [67]:
years_sa = metadata_sa2.loc[metadata_sa2['code'].isin(dataset_sa2.y['code']), ['code', 'year_folder']]
year_lookup_sa = dataset_sa2.y[['code']].merge(years_sa, on='code', how='left')['year_folder'].astype(int).values

train_1516_mask_sa = np.isin(year_lookup_sa, [2015, 2016])
train_17_mask_sa = year_lookup_sa == 2017
test_18_mask_sa = year_lookup_sa == 2018

print(f"S. aureus: 2015-16 n={train_1516_mask_sa.sum()}, 2017 n={train_17_mask_sa.sum()}, 2018 n={test_18_mask_sa.sum()}\n")

X_train_1516_sa = X_sa2[train_1516_mask_sa]
X_2017_sa = X_sa2[train_17_mask_sa]
X_test_18_sa = X_sa2[test_18_mask_sa]

print("S. aureus calibration-tuned threshold results:\n")
print(f"{'Antibiotic':<30} {'AUROC':<8} {'CalibThresh':<12} {'F1@calib':<9} {'Prec@calib':<11} {'Rec@calib':<10} {'TrueRate':<9} {'PredRate@calib'}")

for ab in ANTIBIOTICS_SA:
    y_full = dataset_sa2.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask_sa]
    y_2017 = y_full[train_17_mask_sa]
    y_test_18 = y_full[test_18_mask_sa]

    model = XGBClassifier(random_state=SEED_SA, eval_metric='logloss')
    model.fit(X_train_1516_sa, y_train_1516)

    proba_2017 = model.predict_proba(X_2017_sa)[:, 1]
    observed_rate_2017 = y_2017.mean()

    best_thresh = 0.5
    best_diff = float('inf')
    for t in np.unique(proba_2017):
        pred_rate = (proba_2017 >= t).mean()
        diff = abs(pred_rate - observed_rate_2017)
        if diff < best_diff:
            best_diff = diff
            best_thresh = t

    proba_2018 = model.predict_proba(X_test_18_sa)[:, 1]
    pred_calib = (proba_2018 >= best_thresh).astype(int)

    auroc = roc_auc_score(y_test_18, proba_2018)
    f1_calib = f1_score(y_test_18, pred_calib)
    prec_calib = precision_score(y_test_18, pred_calib, zero_division=0)
    rec_calib = recall_score(y_test_18, pred_calib, zero_division=0)
    true_rate = y_test_18.mean()
    pred_rate_calib = pred_calib.mean()

    print(f"{ab:<30} {auroc:<8.4f} {best_thresh:<12.4f} {f1_calib:<9.4f} {prec_calib:<11.4f} {rec_calib:<10.4f} {true_rate:<9.4f} {pred_rate_calib:.4f}")

S. aureus: 2015-16 n=1070, 2017 n=1318, 2018 n=1066

S. aureus calibration-tuned threshold results:

Antibiotic                     AUROC    CalibThresh  F1@calib  Prec@calib  Rec@calib  TrueRate  PredRate@calib
Penicillin                     0.6610   0.8264       0.7570    0.7682      0.7461     0.7167    0.6961
Erythromycin                   0.5711   0.1273       0.2752    0.2264      0.3509     0.1604    0.2486
Clindamycin                    0.6088   0.1126       0.2882    0.2439      0.3521     0.1332    0.1923
Oxacillin                      0.7426   0.0926       0.4775    0.4306      0.5357     0.1576    0.1961
Ciprofloxacin                  0.6538   0.0165       0.3222    0.2578      0.4296     0.1266    0.2111


In [68]:
years_kp = metadata_kp2.loc[metadata_kp2['code'].isin(dataset_kp2.y['code']), ['code', 'year_folder']]
year_lookup_kp = dataset_kp2.y[['code']].merge(years_kp, on='code', how='left')['year_folder'].astype(int).values

train_1516_mask_kp = np.isin(year_lookup_kp, [2015, 2016])
train_17_mask_kp = year_lookup_kp == 2017
test_18_mask_kp = year_lookup_kp == 2018

print(f"K. pneumoniae: 2015-16 n={train_1516_mask_kp.sum()}, 2017 n={train_17_mask_kp.sum()}, 2018 n={test_18_mask_kp.sum()}\n")

X_train_1516_kp = X_kp2[train_1516_mask_kp]
X_2017_kp = X_kp2[train_17_mask_kp]
X_test_18_kp = X_kp2[test_18_mask_kp]

kleb_panel_final = ['Ceftriaxone', 'Tobramycin', 'Piperacillin-Tazobactam', 'Ciprofloxacin', 'Cotrimoxazole']

print("K. pneumoniae calibration-tuned threshold results:\n")
print(f"{'Antibiotic':<30} {'AUROC':<8} {'CalibThresh':<12} {'F1@calib':<9} {'Prec@calib':<11} {'Rec@calib':<10} {'TrueRate':<9} {'PredRate@calib'}")

for ab in kleb_panel_final:
    y_full = dataset_kp2.to_numpy(ab)
    y_train_1516 = y_full[train_1516_mask_kp]
    y_2017 = y_full[train_17_mask_kp]
    y_test_18 = y_full[test_18_mask_kp]

    model = XGBClassifier(random_state=SEED_KP, eval_metric='logloss')
    model.fit(X_train_1516_kp, y_train_1516)

    proba_2017 = model.predict_proba(X_2017_kp)[:, 1]
    observed_rate_2017 = y_2017.mean()

    best_thresh = 0.5
    best_diff = float('inf')
    for t in np.unique(proba_2017):
        pred_rate = (proba_2017 >= t).mean()
        diff = abs(pred_rate - observed_rate_2017)
        if diff < best_diff:
            best_diff = diff
            best_thresh = t

    proba_2018 = model.predict_proba(X_test_18_kp)[:, 1]
    pred_calib = (proba_2018 >= best_thresh).astype(int)

    auroc = roc_auc_score(y_test_18, proba_2018)
    f1_calib = f1_score(y_test_18, pred_calib)
    prec_calib = precision_score(y_test_18, pred_calib, zero_division=0)
    rec_calib = recall_score(y_test_18, pred_calib, zero_division=0)
    true_rate = y_test_18.mean()
    pred_rate_calib = pred_calib.mean()

    print(f"{ab:<30} {auroc:<8.4f} {best_thresh:<12.4f} {f1_calib:<9.4f} {prec_calib:<11.4f} {rec_calib:<10.4f} {true_rate:<9.4f} {pred_rate_calib:.4f}")

K. pneumoniae: 2015-16 n=759, 2017 n=1225, 2018 n=743

K. pneumoniae calibration-tuned threshold results:

Antibiotic                     AUROC    CalibThresh  F1@calib  Prec@calib  Rec@calib  TrueRate  PredRate@calib
Ceftriaxone                    0.7112   0.0710       0.4021    0.4222      0.3838     0.1332    0.1211
Tobramycin                     0.6901   0.0257       0.2206    0.1923      0.2586     0.0781    0.1050
Piperacillin-Tazobactam        0.7083   0.0609       0.2692    0.2435      0.3011     0.1252    0.1548
Ciprofloxacin                  0.5607   0.0404       0.2437    0.1745      0.4034     0.1602    0.3701
Cotrimoxazole                  0.5672   0.0686       0.2882    0.2552      0.3311     0.1992    0.2584
